<p class="lead">Este suplemento no tiene una clase propia. Existe porque cada unidad, a partir de la Unidad 3, informa un número —una exactitud, un área bajo una curva, un error— y el plan oficial nunca dice de dónde provienen esos números ni cuándo se les puede creer. Las diapositivas de esas unidades son un mapa; este texto es el territorio que hay bajo el número. Léelo de forma lineal una vez, ejecutando cada bloque de código (chunk), y luego úsalo como referencia. Responde cuatro preguntas: cómo estimar el desempeño en pacientes que el modelo nunca ha visto (Secciones 1–4); qué número informar y qué significa clínicamente (Secciones 5–6); cómo las evaluaciones fallan sin que nadie lo note (Sección 7); y qué debe informarse para que otros puedan juzgar el modelo (Sección 8). La Sección 9 reúne las respuestas en una función que reutilizan las unidades posteriores. Esta es la edición en R; respecto de la versión en Python solo cambian el lenguaje y las bibliotecas: las ideas y las referencias son idénticas.</p>

<div class="box key"><h4>Objetivos de aprendizaje</h4>
<ul class="objectives">
<li>Explicar por qué el error medido en la muestra de entrenamiento no es una estimación del error en pacientes nuevos, y definir los roles de las muestras de entrenamiento, validación y prueba.</li>
<li>Implementar a mano en R base la validación cruzada k-fold (aritmética de índices), describir las variantes estratificada, repetida, agrupada y de dejar uno fuera, y explicar por qué la selección de hiperparámetros requiere validación cruzada anidada.</li>
<li>Describir el sobreajuste en términos de la descomposición sesgo–varianza y leer una curva de complejidad y una curva de aprendizaje.</li>
<li>Calcular, a partir de una matriz de confusión, la sensibilidad, la especificidad, la precisión (PPV), el NPV, el F1, la exactitud balanceada y el coeficiente de correlación de Matthews; explicar por qué la exactitud falla ante el desbalance de clases y por qué el PPV depende de la prevalencia.</li>
<li>Construir una curva ROC y una curva de precisión–sensibilidad (precision–recall) a partir de puntajes, calcular el AUC a mano y con <code>pROC</code>, elegir un umbral de decisión a partir de los costos de los dos errores, y evaluar la calibración con un diagrama de confiabilidad y el puntaje de Brier.</li>
<li>Calcular MSE, RMSE, MAE y \(R^2\) para un modelo de regresión y explicar por qué \(R^2\) puede ser negativo en un conjunto de prueba.</li>
<li>Reconocer y prevenir las tres formas clásicas de fuga de datos, y usar divisiones agrupadas cuando los pacientes aportan varios registros.</li>
<li>Distinguir la validación interna de la externa y enumerar lo que la declaración TRIPOD+AI exige que contenga un informe.</li>
</ul></div>

<div class="box bridge"><h4>Cómo leer este notebook</h4>
<p>Cada bloque de código es breve y está comentado. Ejecútalo, lee su salida y luego lee el párrafo que la interpreta. Las cajas marcadas como <em>Ejercicio</em> te piden modificar el bloque anterior y volver a ejecutarlo. Las cajas verdes contienen ideas clave; las naranjas, errores frecuentes; las moradas, vínculos con las unidades que usan este material. Solo se usan R base y los paquetes <code>class</code>, <code>e1071</code>, <code>MASS</code> y <code>pROC</code>, con <code>mlbench</code> para los datos; en un entorno de ejecución de R recién iniciado en Colab, ejecuta una vez <code>install.packages(c("e1071","mlbench","pROC"))</code> y elige <em>Runtime → Change runtime type → R</em>. No se descarga nada. El conjunto de datos recurrente es la muestra Wisconsin Diagnostic Breast Cancer de la Unidad 1 (683 aspirados completos, nueve puntajes de características nucleares, etiqueta de biopsia),<sup class="c"><a href="#r20">[20]</a></sup> provista por el paquete <code>mlbench</code>;<sup class="c"><a href="#r35">[35]</a></sup> los ejemplos de regresión usan los datos de viviendas de Boston (Boston housing),<sup class="c"><a href="#r36">[36]</a></sup> el sustituto en R de los datos de progresión de diabetes de Efron y colaboradores usados en la edición en Python.<sup class="c"><a href="#r21">[21]</a></sup></p></div>



In [ ]:
# Configuración: bibliotecas usadas en todo el notebook. La semilla se fija para que cada
# número citado en el texto sea reproducible.
suppressPackageStartupMessages({
  library(class); library(e1071); library(MASS); library(pROC); library(mlbench)
})
set.seed(0)
BLUE <- "#2b5d9b"; ORANGE <- "#d2691e"; GREEN <- "#3a7d44"; GREY <- "#8a8a8a"
cat("R", as.character(getRversion()),
    "| class", as.character(packageVersion("class")),
    "| e1071", as.character(packageVersion("e1071")),
    "| pROC", as.character(packageVersion("pROC")),
    "| mlbench", as.character(packageVersion("mlbench")), "\n")



La salida impresa registra las versiones con que se construyeron estos apuntes. Una semilla al inicio es el hábito de reproducibilidad más importante (Suplemento S0, Sección 7).

## 1. Por qué la evaluación es la parte más difícil

### 1.1 La pregunta de la generalización

Un modelo supervisado es una función \(f\) ajustada a una muestra de entrenamiento \(\{(\mathbf{x}_i, y_i)\}_{i=1}^{n}\) extraída de una población de pacientes, y usada en pacientes que no están en la muestra. La cantidad que nos interesa es la <strong>pérdida esperada en un caso nuevo</strong>,

<div class="eq">\[ \mathrm{Err} = \mathbb{E}_{(\mathbf{x}, y) \sim P}\big[L\big(y, f(\mathbf{x})\big)\big], \]
<span class="where">donde \(P\) es la distribución poblacional de pacientes, \(L\) es una función de pérdida (pérdida cero–uno para clasificación, error cuadrático para regresión) y la esperanza se toma sobre pacientes que el modelo nunca ha visto. Hastie, Tibshirani y Friedman lo llaman <em>error de prueba</em> o <em>error de generalización</em>.<sup class="c"><a href="#r18">[18]</a></sup></span></div>

No podemos calcular \(\mathrm{Err}\) porque tenemos una muestra, no \(P\). La metodología de evaluación es el problema de estimar \(\mathrm{Err}\) a partir de la muestra sin engañarnos a nosotros mismos, y es más difícil que ajustar el modelo, porque el modelo se eligió para que la muestra se viera bien.

### 1.2 El error de entrenamiento no es una medición

La estimación obvia es el error en la propia muestra de entrenamiento, el <strong>error de entrenamiento</strong> (o <em>error de resustitución</em>). Está sesgado hacia abajo, tanto más cuanto más flexible es el modelo, porque los mismos datos se usaron dos veces: para elegir \(f\) y para juzgarla. El caso extremo no necesita estadística: el clasificador del vecino más cercano (1-nearest-neighbour, Unidad 3) predice para un caso la etiqueta del caso de entrenamiento más cercano; en la muestra de entrenamiento el caso más cercano es el propio caso, de modo que la exactitud de entrenamiento es esencialmente 100 % por construcción, sean cuales sean los datos.



In [ ]:
# Exactitud de entrenamiento frente a exactitud en casos reservados, para el clasificador del vecino más cercano (1-NN).
data(BreastCancer, package = "mlbench")
bc <- BreastCancer[, -1]                                   # elimina la columna Id
bc <- bc[complete.cases(bc), ]                             # elimina las 16 filas con valores faltantes
bc[, 1:9] <- lapply(bc[, 1:9], function(z) as.numeric(as.character(z)))  # factor ordinal -> numérico
X <- as.matrix(bc[, 1:9])                                  # nueve puntajes de características nucleares (1-10)
feature_names <- colnames(X)
y_pos <- as.integer(bc$Class == "malignant")               # 1 = maligno (condición que se busca detectar), 0 = benigno
Xs <- scale(X)                                             # características estandarizadas, reutilizadas más abajo

set.seed(0)
idx <- sample(nrow(X), round(0.7 * nrow(X)))               # 70% entrenamiento / 30% prueba
ctr <- scale(X[idx, ])
cte <- scale(X[-idx, ], center = attr(ctr, "scaled:center"), scale = attr(ctr, "scaled:scale"))
tr_pred <- knn(ctr, ctr, factor(y_pos[idx]), k = 1)
te_pred <- knn(ctr, cte, factor(y_pos[idx]), k = 1)
cat(sprintf("training cases: %d, held-out cases: %d\n", length(idx), nrow(X) - length(idx)))
cat(sprintf("1-NN accuracy on the training sample : %.3f\n", mean(tr_pred == y_pos[idx])))
cat(sprintf("1-NN accuracy on the held-out sample : %.3f\n", mean(te_pred == y_pos[-idx])))



La exactitud de entrenamiento es 1.000 y no dice nada; la exactitud en los casos reservados, cercana a 0.95, es el primer número de este notebook que merece el nombre de <em>medición</em>. La diferencia es el <strong>optimismo</strong> del error de entrenamiento: máximo para 1-NN, pequeño para un modelo lineal con pocos parámetros, potencialmente total de nuevo para una red con millones de parámetros, y nunca cero. (Una salvedad propia de este conjunto de datos: sus nueve características son puntajes enteros de 1–10, de modo que algunos aspirados comparten un vector de características idéntico; <code>knn</code> resuelve al azar los empates de distancia resultantes, y por eso no conviene leer en la cifra de entrenamiento más que "esencialmente perfecta").

### 1.3 Tres muestras, tres roles

En la práctica, los datos se separan en hasta tres muestras disjuntas, cada una con un rol.

| Muestra | Rol | ¿Cuántas veces se usa? |
|---|---|---|
| **Entrenamiento** | Ajustar los parámetros del modelo (los pesos, las divisiones del árbol, los vecinos almacenados) | Tantas como sea necesario |
| **Validación** (desarrollo, ajuste) | Elegir entre modelos e <em>hiperparámetros</em> —los parámetros de configuración que el algoritmo de ajuste no aprende, como \(k\) en k-NN, la profundidad de un árbol, la intensidad de la regularización | Tantas como sea necesario, pero cada consulta cuesta un poco de optimismo |
| **Prueba** (reservada, evaluación) | Estimar el error de generalización del modelo final | **Una vez.** Un conjunto de prueba consultado dos veces se ha convertido en un conjunto de validación |

Las proporciones habituales son 60–20–20, 70–15–15 u 80–10–10 (entrenamiento–validación–prueba). No hay una regla universal: una mayor proporción de entrenamiento da un mejor modelo, una mayor proporción de prueba una medición más precisa (Sección 2), y con conjuntos de datos muy grandes un conjunto de prueba del 1 % ya contiene más casos que un estudio pequeño. La tercera fila de la tabla es la regla que más a menudo se rompe: si el conjunto de prueba se usa para comparar diez modelos y se informa el mejor, el número es el máximo de diez estimaciones ruidosas y está sesgado hacia arriba —los datos se usaron de nuevo dos veces, para elegir y para juzgar (la Sección 3.4 lo mide). Cuando los datos son escasos, la validación cruzada dentro de la muestra de entrenamiento cumple el rol de validación (Sección 3) y el conjunto de prueba se guarda para el final.



In [ ]:
# Esquema de las tres muestras y sus roles.
par(mar = c(0, 0, 0, 0))
plot(NA, xlim = c(0, 100), ylim = c(-0.4, 2.7), axes = FALSE, xlab = "", ylab = "")
blocks <- list(c(0, 60), c(60, 20), c(80, 20)); cols <- c(BLUE, GREEN, ORANGE)
labs <- c("TRAINING\n(fit parameters)", "VALIDATION\n(choose model &\nhyperparameters)", "TEST\n(estimate error,\nused once)")
for (i in 1:3) { x0 <- blocks[[i]][1]; w <- blocks[[i]][2]
  rect(x0, 1, x0 + w, 2, col = adjustcolor(cols[i], 0.85), border = NA)
  text(x0 + w / 2, 1.5, labs[i], col = "white", font = 2, cex = 0.85) }
arrows(0, 0.55, 60, 0.55, code = 3, length = 0.08, col = GREY); text(30, 0.25, "many fits, many looks", cex = 0.85, col = GREY)
arrows(60, 0.55, 80, 0.55, code = 3, length = 0.08, col = GREY); text(70, 0.22, "many looks\n(each costs optimism)", cex = 0.8, col = GREY)
arrows(80, 0.55, 100, 0.55, code = 3, length = 0.08, col = ORANGE); text(90, 0.25, "ONE look", cex = 0.85, col = ORANGE, font = 2)
text(50, 2.4, "all available labelled cases (one random, stratified shuffle)", cex = 0.95)



<p class="caption"><b>Figura 1.</b> Las tres muestras disjuntas y sus roles. Las proporciones (aquí 60/20/20) son convencionales, no sagradas; lo que importa es que los parámetros nunca se ajusten en la muestra de validación ni en la de prueba, y que la muestra de prueba se examine una sola vez, después de haber tomado todas las decisiones.</p>

<div class="box warn"><h4>Error frecuente</h4><p>"Nuestro modelo alcanzó 99 % de exactitud" no es un resultado hasta responder tres preguntas: en qué casos (entrenamiento, validación o prueba), separados cómo (división aleatoria, por hospital, por fecha) y cuántos (un conjunto de prueba de 20 pacientes no puede distinguir 99 % de 90 %). Todo número que aparezca más adelante en este curso debe leerse con esas tres preguntas en mente.</p></div>

## 2. El método hold-out y su varianza

### 2.1 Una sola división es un solo sorteo

La estimación honesta más simple es el método <strong>hold-out</strong>: dividir al azar en una parte de entrenamiento y una parte de prueba, ajustar en la primera y evaluar en la segunda. Su debilidad es que la estimación depende de qué casos cayeron en la parte de prueba. Con \(n_{\text{test}}\) casos de prueba y exactitud verdadera \(p\), el número de predicciones correctas es aproximadamente binomial, de modo que

<div class="eq">\[ \mathrm{SE}(\hat{p}) \approx \sqrt{\frac{\hat{p}(1-\hat{p})}{n_{\text{test}}}}, \]
<span class="where">que para \(\hat{p} = 0.96\) y \(n_{\text{test}} \approx 205\) es aproximadamente 0.014: dos modelos cuyas exactitudes hold-out difieren en un punto sobre 205 casos no son distinguibles. El estudio de Kohavi de 1995 es la comparación empírica clásica entre el método hold-out, la validación cruzada y el bootstrap, y el origen de la recomendación convencional de la validación cruzada estratificada de diez particiones.<sup class="c"><a href="#r2">[2]</a></sup></span></div>

El experimento repite la división 70/30 200 veces con semillas distintas, ajustando cada vez el mismo clasificador de 5 vecinos más cercanos.



In [ ]:
# Hold-out repetido: 200 divisiones aleatorias 70/30 de los mismos datos y el mismo modelo (5-NN sobre características estandarizadas).
accs <- sapply(0:199, function(s) {
  set.seed(s); id <- sample(nrow(X), round(0.7 * nrow(X)))     # aquí sin estratificación, a propósito
  tr <- scale(X[id, ]); te <- scale(X[-id, ], center = attr(tr, "scaled:center"), scale = attr(tr, "scaled:scale"))
  mean(knn(tr, te, factor(y_pos[id]), k = 5) == y_pos[-id])
})
n_test <- nrow(X) - round(0.7 * nrow(X))
se_bin <- sqrt(mean(accs) * (1 - mean(accs)) / n_test)
cat(sprintf("200 hold-out accuracies: mean %.3f, SD %.3f, min %.3f, max %.3f\n", mean(accs), sd(accs), min(accs), max(accs)))
cat(sprintf("binomial standard error predicted for n_test = %d: %.3f\n", n_test, se_bin))
hist(accs, breaks = 18, col = BLUE, border = "white", xlab = "test accuracy of the same model on a different random 30% test set",
     ylab = "number of splits", main = "The hold-out estimate is a random variable")
abline(v = mean(accs), col = ORANGE, lwd = 2); abline(v = accs[1], col = GREEN, lwd = 2, lty = 2)
legend("topleft", bty = "n", cex = 0.8, lty = c(1, 2), lwd = 2, col = c(ORANGE, GREEN),
       legend = c(sprintf("mean = %.3f", mean(accs)), sprintf("the split with seed 0: %.3f", accs[1])))



<p class="caption"><b>Figura 2.</b> Distribución de la exactitud de prueba de un modelo fijo en 200 divisiones hold-out aleatorias. La dispersión (desviación estándar de aproximadamente 0.011) se acerca a lo que predice la fórmula binomial (aproximadamente 0.012) para 205 casos de prueba, y el rango entre la peor y la mejor división abarca varios puntos porcentuales. La línea discontinua es el único número que habría informado un notebook con <code>set.seed(0)</code>.</p>

Una exactitud hold-out es un sorteo de una distribución cuyo ancho está determinado por el tamaño de prueba. Informarla sin el tamaño de prueba, o comparar dos modelos en divisiones distintas, es informar ruido. De ello se siguen dos remedios: evaluar en todos los datos por rotación (validación cruzada, Sección 3) e informar un intervalo en lugar de un punto.

### 2.2 Estratificación

Una división aleatoria también puede, por azar, dejar proporciones de clases distintas en la parte de prueba y en la de entrenamiento, lo que sesga la estimación y aumenta su varianza. La división <strong>estratificada</strong> realiza la división dentro de cada clase, de modo que las proporciones se conservan. Con 683 casos el efecto es pequeño; con los tamaños muestrales de un estudio piloto clínico no lo es, y una clase puede desaparecer por completo de la parte de prueba. El bloque de código reduce el conjunto de prueba a 20 casos para que el efecto sea visible.



In [ ]:
# Estratificación: la fracción de casos malignos en un conjunto de prueba de 20 casos, con y sin estratificación, en 300 divisiones.
malignant_fraction <- function(stratified, n_splits = 300, test_size = 20) {
  sapply(0:(n_splits - 1), function(s) {
    set.seed(s)
    if (stratified) {
      n_pos <- round(test_size * mean(y_pos == 1))
      te <- c(sample(which(y_pos == 1), n_pos), sample(which(y_pos == 0), test_size - n_pos))
    } else te <- sample(nrow(X), test_size)
    mean(y_pos[te] == 1)
  })
}
plain <- malignant_fraction(FALSE); strat <- malignant_fraction(TRUE)
tab <- data.frame(min = c(min(plain), min(strat)), mean = c(mean(plain), mean(strat)), max = c(max(plain), max(strat)),
                  SD = c(sd(plain), sd(strat)), `splits with < 20% malignant` = c(sum(plain < 0.2), sum(strat < 0.2)),
                  row.names = c("random split", "stratified split"), check.names = FALSE)
cat(sprintf("fraction of malignant cases in the whole sample: %.3f\n", mean(y_pos == 1)))
round(tab, 3)



Sin estratificación, la fracción maligna de un conjunto de prueba de 20 casos oscila ampliamente de una división a otra; con ella, cada división contiene el mismo número de casos malignos, de modo que las métricas por clase de la Sección 5 tienen un denominador estable. La estratificación no cuesta nada y es la opción por defecto para clasificación en este curso. Para regresión, la precaución análoga es comprobar que los rangos de la variable objetivo en entrenamiento y en prueba se superpongan.

<div class="box"><h4>Ejercicio 1</h4><p>En el bloque de hold-out repetido, cambia la fracción de entrenamiento de 0.7 a 0.9 y luego a 0.5, y observa la desviación estándar de las 200 exactitudes. Un conjunto de prueba más pequeño hace más ruidosa la <em>estimación</em>; un conjunto de entrenamiento más pequeño empeora el <em>modelo</em>. ¿Cuál de los dos efectos domina en cada extremo? Esa disyuntiva es la razón por la que existe la validación cruzada.</p></div>

## 3. Validación cruzada

### 3.1 La idea

La <strong>validación cruzada k-fold</strong> (k-fold cross-validation, CV) resuelve por rotación la disyuntiva del Ejercicio 1. La muestra se divide en \(k\) particiones (folds) disjuntas de tamaño casi igual; para cada partición \(j\), el modelo se ajusta en las otras \(k-1\) particiones y se evalúa en la partición \(j\). Cada caso se evalúa exactamente una vez, por un modelo que no lo vio. La estimación por CV es el promedio de los \(k\) puntajes de las particiones,

<div class="eq">\[ \widehat{\mathrm{Err}}_{\text{CV}} = \frac{1}{k} \sum_{j=1}^{k} \mathrm{Err}_j, \qquad \mathrm{Err}_j = \frac{1}{|F_j|} \sum_{i \in F_j} L\big(y_i, f^{(-j)}(\mathbf{x}_i)\big), \]
<span class="where">donde \(F_j\) es el conjunto de índices de la partición \(j\) y \(f^{(-j)}\) el modelo ajustado sin ella. Stone formalizó el método en 1974 y demostró lo que la Sección 3.4 ilustra: los mismos datos no pueden a la vez seleccionar un modelo y evaluarlo.<sup class="c"><a href="#r1">[1]</a></sup> Lo convencional es \(k = 5\) o \(10\); \(k = n\) corresponde a <em>dejar uno fuera</em> (leave-one-out, LOO).</span></div>

Observa qué estima la CV: cada \(f^{(-j)}\) se entrena con \((k-1)/k\) de los datos, de modo que la CV estima el error del <em>procedimiento</em> con aproximadamente ese tamaño muestral, no el del modelo final ajustado con todos los datos, que se espera sea algo mejor. El número de la CV es, por tanto, ligeramente conservador —la dirección correcta en la que equivocarse.

### 3.2 Ejemplo resuelto a mano

Toma \(n = 10\) casos con índices 1–10 y \(k = 5\). Sin barajar, bloques consecutivos de \(n/k = 2\) casos pasan a ser cada partición de prueba:

| partición \(j\) | índices de prueba \(F_j\) | índices de entrenamiento | modelo ajustado | evaluado en |
|---|---|---|---|---|
| 1 | {1, 2} | {3, 4, 5, 6, 7, 8, 9, 10} | \(f^{(-1)}\) | casos 1, 2 |
| 2 | {3, 4} | {1, 2, 5, 6, 7, 8, 9, 10} | \(f^{(-2)}\) | casos 3, 4 |
| 3 | {5, 6} | {1, 2, 3, 4, 7, 8, 9, 10} | \(f^{(-3)}\) | casos 5, 6 |
| 4 | {7, 8} | {1, 2, 3, 4, 5, 6, 9, 10} | \(f^{(-4)}\) | casos 7, 8 |
| 5 | {9, 10} | {1, 2, 3, 4, 5, 6, 7, 8} | \(f^{(-5)}\) | casos 9, 10 |

Supón que las exactitudes de las particiones son 1.0, 0.5, 1.0, 1.0 y 0.5. La exactitud de CV es su promedio, \(0.80\), que aquí coincide con el agregado \(8/10\) porque las particiones son de igual tamaño. Si los casos estuvieran ordenados —todos los malignos primero, como ocurre cuando un registro se exporta ordenado por diagnóstico—, las particiones sin barajar serían catastróficas; por eso se usa un barajado aleatorio con una semilla fija siempre que el orden de las filas pueda contener información. El bloque de código construye las particiones con aritmética de índices, las contrasta con la tabla y aplica el procedimiento a los 683 casos de cáncer de mama.



In [ ]:
# Validación cruzada de 5 particiones a mano: solo aritmética de índices, sin splitter de biblioteca.
kfold_indices <- function(n, k, shuffle = FALSE, seed = 0) {
  if (shuffle) { set.seed(seed); ord <- sample(n) } else ord <- seq_len(n)
  blocks <- split(ord, cut(seq_len(n), breaks = k, labels = FALSE))   # k bloques de tamaño casi igual
  lapply(blocks, function(f) list(train = setdiff(ord, f), test = sort(f)))
}
# (a) el ejemplo de juguete de la tabla: n = 10, k = 5, sin barajar
toy <- kfold_indices(10, 5)
for (j in seq_along(toy)) cat(sprintf("fold %d: test %-7s train %s\n", j,
    paste(toy[[j]]$test, collapse = " "), paste(toy[[j]]$train, collapse = " ")))

# (b) el mismo procedimiento sobre los datos de cáncer de mama, con barajado, para el modelo 5-NN
folds5 <- kfold_indices(nrow(X), 5, shuffle = TRUE, seed = 0)
fold_acc <- sapply(folds5, function(f) mean(knn(Xs[f$train, ], Xs[f$test, ], factor(y_pos[f$train]), k = 5) == y_pos[f$test]))
cat("\nbreast cancer, 5 folds of", sapply(folds5, function(f) length(f$test)), "cases\n")
cat("fold accuracies:", round(fold_acc, 3), " ->  CV accuracy =", round(mean(fold_acc), 3),
    sprintf("(SD across folds %.3f)", sd(fold_acc)), "\n")



La parte (a) reproduce exactamente la tabla. La parte (b) da una exactitud de CV de cinco particiones para 5-NN de aproximadamente 0.96–0.97, con diferencias de uno o dos puntos entre particiones —la dispersión de la Figura 2, ahora visible dentro de una sola ejecución, y en sí misma un hallazgo: un modelo cuyos puntajes por partición fueran de 0.80 a 0.99 sería inestable.

### 3.3 Los esquemas de remuestreo

Una función de conveniencia como <code>cross_val_score</code> de scikit-learn<sup class="c"><a href="#r22">[22]</a></sup> encapsula este ciclo; en R el mismo efecto se logra con un <code>sapply</code> de una línea sobre una lista de índices, y el paquete <code>caret</code><sup class="c"><a href="#r34">[34]</a></sup> ofrece un equivalente empaquetado (<code>trainControl</code>, <code>train</code>) para quien lo prefiera. Los esquemas difieren solo en cómo se construye la lista de índices:

- **k-fold simple** — la versión de la Sección 3.2; adecuada para conjuntos de datos grandes y balanceados, donde cada partición es representativa por puro tamaño. Costo: \(k\) ajustes.
- **k-fold estratificado** — cada partición conserva las proporciones de clases (Sección 2.2); la opción para conjuntos de datos pequeños o desbalanceados, y la opción por defecto para clasificación en este curso. Costo: \(k\) ajustes.
- **k-fold estratificado repetido** — el procedimiento k-fold repetido \(r\) veces con distintos barajados, promediando \(kr\) puntajes. Reduce la varianza debida a la partición particular (no la varianza debida a tener solo \(n\) casos, que nada reduce). Costo: \(kr\) ajustes.
- **Dejar uno fuera** — \(k = n\): cada caso es predicho por un modelo entrenado con todos los demás. Casi insesgado para el error con tamaño \(n-1\), pero cuesta \(n\) ajustes y, como los \(n\) conjuntos de entrenamiento son casi idénticos, su estimación tiene alta varianza;<sup class="c"><a href="#r18">[18]</a></sup> adecuado solo para conjuntos de datos muy pequeños.
- **k-fold agrupado** — los casos de un mismo grupo (paciente, hospital, serie de imágenes) nunca aparecen a la vez en particiones de entrenamiento y de prueba (Sección 7.3).
- **Shuffle-split (hold-out repetido)** — como en la Figura 2; los conjuntos de prueba se superponen entre repeticiones.

El tutorial de Raschka revisa estos esquemas, y las alternativas bootstrap omitidas aquí, con código resuelto.<sup class="c"><a href="#r19">[19]</a></sup> El bloque de código los compara sobre el mismo modelo y los mismos datos, junto con el número de ajustes que cuesta cada uno.



In [ ]:
# El mismo modelo (5-NN) evaluado con varios esquemas de remuestreo; media, SD entre particiones y número de ajustes.
strat_folds <- function(y, k, seed = 0) {                    # asigna una partición a cada caso, conservando las proporciones de clases
  set.seed(seed); fold <- integer(length(y))
  for (cl in unique(y)) { i <- sample(which(y == cl)); fold[i] <- (seq_along(i) - 1) %% k + 1 }
  fold
}
strat_kfold <- function(y, k, seed = 0) { f <- strat_folds(y, k, seed)
  lapply(1:k, function(j) list(train = which(f != j), test = which(f == j))) }
acc_of <- function(splits) sapply(splits, function(f) mean(knn(Xs[f$train, ], Xs[f$test, ], factor(y_pos[f$train]), k = 5) == y_pos[f$test]))

schemes <- list(
  "KFold(5, shuffle)"             = kfold_indices(nrow(X), 5, shuffle = TRUE, seed = 0),
  "StratifiedKFold(5)"            = strat_kfold(y_pos, 5, 0),
  "StratifiedKFold(10)"           = strat_kfold(y_pos, 10, 0),
  "RepeatedStratifiedKFold(5x10)" = do.call(c, lapply(1:10, function(r) strat_kfold(y_pos, 5, r))),
  "ShuffleSplit(20 x 30%)"        = lapply(1:20, function(s) { set.seed(s); id <- sample(nrow(X), round(0.7 * nrow(X))); list(train = id, test = setdiff(seq_len(nrow(X)), id)) }),
  "LeaveOneOut (683 fits)"        = lapply(seq_len(nrow(X)), function(i) list(train = setdiff(seq_len(nrow(X)), i), test = i)))
rows <- lapply(names(schemes), function(nm) { t0 <- Sys.time(); s <- acc_of(schemes[[nm]])
  data.frame(scheme = nm, `mean accuracy` = mean(s), `SD across folds` = sd(s), fits = length(s),
             seconds = round(as.numeric(Sys.time() - t0), 2), check.names = FALSE) })
r <- do.call(rbind, rows); rownames(r) <- NULL; r[, 2:3] <- round(r[, 2:3], 3); r



Todos los esquemas coinciden en la media con una diferencia de pocas milésimas: con 683 casos y un modelo estable, la estimación no es sensible al esquema. La SD de dejar uno fuera carece de sentido como incertidumbre (cada partición es un caso puntuado 0 o 1); la columna de costo es la lista de precios —instantáneo para k-NN, decisivo para las redes de las Unidades 6 y 8, donde un ajuste puede tardar horas. La figura hace visibles los esquemas: cada fila un esquema, cada columna un caso (ordenado por clase), el color la partición en que se evalúa el caso.



In [ ]:
# ¿Qué partición evalúa qué caso? Una comparación visual de cuatro esquemas en una muestra pequeña ordenada.
n_small <- 40; y_small <- c(rep(1, 14), rep(0, 26)); g_small <- rep(1:10, each = 4)   # 14 malignos y luego 26 benignos; 10 pacientes x 4 registros
assign_fold <- function(scheme) {
  if (scheme == "kf")  return(cut(1:n_small, 4, labels = FALSE))
  if (scheme == "kfs") { set.seed(0); ord <- sample(n_small); fo <- integer(n_small); fo[ord] <- cut(1:n_small, 4, labels = FALSE); return(fo) }
  if (scheme == "skf") return(strat_folds(y_small, 4, 0))
  if (scheme == "gkf") { set.seed(0); pf <- sample(rep(1:4, length.out = 10)); return(pf[g_small]) }
}
nms <- c("KFold(4), no shuffle" = "kf", "KFold(4), shuffle" = "kfs", "StratifiedKFold(4)" = "skf", "GroupKFold(4), 10 patients" = "gkf")
cols4 <- c(BLUE, ORANGE, GREEN, GREY)
par(mfrow = c(5, 1), mar = c(0.4, 11, 0.4, 1), oma = c(3, 0, 2, 0))
for (nm in names(nms)) { image(1:n_small, 1, matrix(assign_fold(nms[nm]), ncol = 1), col = cols4, axes = FALSE, xlab = "", ylab = "")
  mtext(nm, side = 2, las = 1, cex = 0.72, line = 0.5) }
image(1:n_small, 1, matrix(y_small, ncol = 1), col = c("grey80", "grey30"), axes = FALSE, xlab = "", ylab = "")
mtext("class (dark = malignant)", side = 2, las = 1, cex = 0.68, line = 0.5)
mtext("case index (sorted by class);  colour = fold in which the case is tested", side = 1, outer = TRUE, line = 1, cex = 0.78)
mtext("Fold assignment of 40 cases under four schemes", side = 3, outer = TRUE, font = 2, cex = 0.9)



<p class="caption"><b>Figura 3.</b> Pertenencia a particiones de 40 casos (14 malignos y luego 26 benignos) con cuatro esquemas. El k-fold sin barajar pone los primeros casos —todos malignos— en una misma partición de prueba. El k-fold con barajado mezcla las clases, pero los recuentos por partición siguen variando. El k-fold estratificado da a cada partición la misma composición de clases. El k-fold agrupado mantiene juntos los cuatro registros de cada paciente, de modo que ningún paciente queda a ambos lados de la frontera entre entrenamiento y prueba (Sección 7.3).</p>

### 3.4 Validación cruzada anidada y el optimismo de seleccionar en el conjunto de prueba

La validación cruzada es también la herramienta estándar para <strong>elegir hiperparámetros</strong>: calcular el puntaje de CV de cada configuración candidata y quedarse con la mejor. El peligro es informar ese mejor puntaje como el desempeño del modelo: es el máximo de muchas estimaciones ruidosas y está sesgado hacia arriba, tanto más cuantos más candidatos haya. Varma y Simon mostraron en conjuntos de datos genómicos pequeños que el sesgo puede superar los 20 puntos porcentuales; el remedio es la <strong>validación cruzada anidada</strong>: un ciclo externo reserva una partición de prueba y, dentro de cada conjunto de entrenamiento externo, una CV interna completa selecciona los hiperparámetros, de modo que las particiones de prueba externas nunca se usan para seleccionar.<sup class="c"><a href="#r3">[3]</a></sup> Cawley y Talbot ofrecen la teoría general.<sup class="c"><a href="#r23">[23]</a></sup>

La demostración más limpia usa datos <em>sin ninguna señal</em> —100 casos, 20 características de ruido, etiquetas aleatorias—, donde toda estimación honesta debe rondar 0.50. El bloque de código ajusta una máquina de vectores de soporte (Unidad 3) sobre una grilla de 5 × 5 e informa (i) el mejor puntaje de CV interna, que un análisis descuidado publicaría, y (ii) la estimación anidada.



In [ ]:
# Sesgo optimista de la selección de hiperparámetros, medido en ruido puro (exactitud verdadera = 0.5 por construcción).
Cs <- 10^seq(-2, 2, length.out = 5); gs <- 10^seq(-3, 1, length.out = 5)    # 25 configuraciones candidatas
grid_best <- function(Xd, yd, seed) {                                      # mejor exactitud de CV de 5 particiones en la grilla
  fi <- strat_folds(as.integer(yd), 5, seed); best <- -1
  for (C in Cs) for (g in gs) {
    a <- sapply(1:5, function(j) { tr <- fi != j
      mean(predict(svm(Xd[tr, ], yd[tr], cost = C, gamma = g, kernel = "radial"), Xd[!tr, ]) == yd[!tr]) })
    if (mean(a) > best) { best <- mean(a); bp <- c(C, g) } }
  list(best = best, par = bp)
}
naive <- nested <- numeric(6)
for (rep in 1:6) {                                                         # 6 conjuntos de datos de ruido independientes
  set.seed(rep); Xn <- matrix(rnorm(100 * 20), 100, 20); yn <- factor(sample(0:1, 100, replace = TRUE))
  naive[rep] <- grid_best(Xn, yn, rep)$best                               # mejor de 25 puntajes de CV interna: sesgado
  of <- strat_folds(as.integer(yn), 5, 100 + rep)                         # particiones externas
  oacc <- sapply(1:5, function(j) { tr <- of != j; bp <- grid_best(Xn[tr, ], yn[tr], 1000 + j)$par
    mean(predict(svm(Xn[tr, ], yn[tr], cost = bp[1], gamma = bp[2], kernel = "radial"), Xn[!tr, ]) == yn[!tr]) })
  nested[rep] <- mean(oacc)                                               # honesto
}
cat(sprintf("best inner-CV score (naive): mean %.3f  [%.3f, %.3f]\n", mean(naive), min(naive), max(naive)))
cat(sprintf("nested-CV estimate:          mean %.3f  [%.3f, %.3f]\n", mean(nested), min(nested), max(nested)))
cat(sprintf("mean optimism of the naive number: %+.3f  (true accuracy is 0.500)\n", mean(naive) - mean(nested)))



En datos donde no hay nada que aprender, el mejor de los 25 puntajes internos promedia varios puntos por encima de 0.5 —una señal fabricada enteramente por la selección—, mientras que la estimación anidada se mantiene cerca del azar. Con 100 casos el efecto es moderado; con los 20–50 casos por clase de un estudio típico de biomarcadores y una grilla de cientos de configuraciones, la cifra de 20 puntos de Varma y Simon se alcanza con facilidad, y un modelo inútil se publica como útil.<sup class="c"><a href="#r3">[3]</a></sup>

<div class="box key"><h4>Idea clave: un nivel de mirar, un nivel de reservar</h4><p>Toda decisión que se tome mirando un puntaje —qué modelo, qué hiperparámetros, qué características, qué umbral— debe tomarse con datos que luego no se usen para informar el puntaje. La CV anidada automatiza esto para los hiperparámetros; la misma lógica se aplica a la selección de características (Sección 7.1) y a la exploración manual: si probaste veinte cosas e informas la mejor, el número es optimista, y un conjunto de prueba nuevo es el único remedio.</p></div>

<div class="box"><h4>Ejercicio 2</h4><p>Amplía la grilla a 10 × 10 configuraciones y reduce la muestra a 50 casos: ¿el optimismo crece o disminuye? Luego reemplaza el ruido por los datos de cáncer de mama: con señal real, ¿qué tan grande es la diferencia entre el número ingenuo y el anidado?</p></div>

## 4. Sobreajuste y la disyuntiva sesgo–varianza

### 4.1 Curvas de complejidad

La <strong>complejidad</strong> de un modelo es, a grandes rasgos, el número de parámetros libres que puede ajustar: el grado de un polinomio (el grado \(d\) tiene \(d+1\) coeficientes), la profundidad de un árbol, el número de pesos de una red. Un modelo presenta <strong>subajuste</strong> cuando es demasiado simple para representar la regularidad de los datos —una recta trazada a través de una relación curva— y se desempeña mal tanto en los datos de entrenamiento como en los de prueba. Presenta <strong>sobreajuste</strong> cuando es lo bastante complejo para reproducir los accidentes de la muestra de entrenamiento —ruido en las etiquetas, correlaciones casuales— en lugar de la regularidad; su señal característica es una brecha creciente entre el desempeño de entrenamiento y el de los casos reservados a medida que aumenta la flexibilidad. En k vecinos más cercanos (Unidad 3) la flexibilidad la fija \(k\): \(k = 1\) sigue individualmente cada caso de entrenamiento, \(k = n\) predice la clase mayoritaria para todos. El bloque de código recorre \(k\) y registra la exactitud de entrenamiento y la de CV de diez particiones.



In [ ]:
# Curva de complejidad: exactitud de entrenamiento y exactitud de validación cruzada de k-NN en función de k.
ks <- c(1, 3, 5, 7, 9, 11, 15, 21, 31, 41, 61, 81, 101, 151, 201, 301)
cv10 <- strat_kfold(y_pos, 10, 0)
train_acc <- sapply(ks, function(k) mean(knn(Xs, Xs, factor(y_pos), k = k) == y_pos))
cv_acc <- sapply(ks, function(k) mean(sapply(cv10, function(f) mean(knn(Xs[f$train, ], Xs[f$test, ], factor(y_pos[f$train]), k = k) == y_pos[f$test]))))
best <- ks[which.max(cv_acc)]
cat(sprintf("best k by CV: %d (CV accuracy %.3f); at k=1: train %.3f / CV %.3f; at k=301: train %.3f / CV %.3f\n",
            best, max(cv_acc), train_acc[1], cv_acc[1], tail(train_acc, 1), tail(cv_acc, 1)))
plot(ks, train_acc, type = "o", pch = 19, col = GREY, log = "x", xlim = rev(range(ks)), ylim = c(0.80, 1.01),
     xlab = "k (number of neighbours) - flexibility increases to the right", ylab = "accuracy",
     main = "k-NN on the breast-cancer data: training vs cross-validated accuracy")
lines(ks, cv_acc, type = "o", pch = 15, col = BLUE); abline(v = best, col = ORANGE, lty = 2)
legend("bottom", bty = "n", cex = 0.8, pch = c(19, 15, NA), lty = c(1, 1, 2), col = c(GREY, BLUE, ORANGE),
       legend = c("training accuracy (resubstitution)", "10-fold CV accuracy", sprintf("best k by CV = %d", best)))



<p class="caption"><b>Figura 4.</b> Exactitud de entrenamiento y de validación cruzada de k-NN en función de \(k\) (la flexibilidad aumenta hacia la derecha). En el borde derecho la exactitud de entrenamiento es 1.000 y la de CV es menor: el modelo memoriza. En el borde izquierdo ambas caen juntas hacia la tasa de la clase mayoritaria, 0.650 (444 benignos de 683): subajuste. La curva de CV alcanza su máximo entre ambos extremos, donde las curvas están próximas. La curva de entrenamiento es monótona y no informativa; solo la curva de CV puede localizar el óptimo.</p>

### 4.2 Curvas de aprendizaje

Un segundo diagnóstico grafica el desempeño en función del <em>tamaño de la muestra de entrenamiento</em>. El puntaje de entrenamiento baja a medida que \(n\) crece (una muestra más grande es más difícil de memorizar) y el puntaje de validación sube (hay más de qué aprender). La brecha en el mayor \(n\) es la varianza restante; el nivel al que convergen las curvas es el sesgo restante. Una curva de aprendizaje responde la pregunta más cara del trabajo aplicado —¿ayudaría reclutar más pacientes?— antes de reclutarlos.



In [ ]:
# Curvas de aprendizaje de dos modelos de distinta flexibilidad: puntaje de entrenamiento y puntaje de CV frente al tamaño del conjunto de entrenamiento.
dff <- data.frame(Xs, y = y_pos)
pred_logit <- function(sub, test) { fit <- suppressWarnings(glm(y ~ ., dff[sub, ], family = binomial)); as.integer(predict(fit, dff[test, ], type = "response") >= 0.5) }
pred_1nn   <- function(sub, test) as.integer(as.character(knn(Xs[sub, , drop = FALSE], Xs[test, , drop = FALSE], factor(y_pos[sub]), k = 1)))
fracs <- seq(0.1, 1, length.out = 8); cv <- strat_kfold(y_pos, 5, 0)
learn <- function(predf) t(sapply(fracs, function(fr) {
  tr <- cv_ <- numeric(length(cv))
  for (i in seq_along(cv)) { f <- cv[[i]]; m <- max(15, round(fr * length(f$train))); set.seed(i); sub <- f$train[sample(length(f$train), m)]
    cv_[i] <- mean(predf(sub, f$test) == y_pos[f$test]); tr[i] <- mean(predf(sub, sub) == y_pos[sub]) }
  c(size = round(fr * length(cv[[1]]$train)), train = mean(tr), cv = mean(cv_)) }))
par(mfrow = c(1, 2))
for (nm in c("logistic regression (Unit 2)", "1-nearest neighbour")) {
  L <- if (grepl("logistic", nm)) learn(pred_logit) else learn(pred_1nn)
  plot(L[, "size"], L[, "train"], type = "o", pch = 19, col = GREY, ylim = c(0.85, 1.005), xlab = "number of training cases", ylab = "accuracy", main = nm)
  lines(L[, "size"], L[, "cv"], type = "o", pch = 15, col = BLUE)
  if (grepl("logistic", nm)) legend("bottomright", bty = "n", cex = 0.8, pch = c(19, 15), col = c(GREY, BLUE), legend = c("training score", "CV score"))
  cat(sprintf("%-30s at n=%d: train %.3f, CV %.3f, gap %.3f\n", nm, L[nrow(L), "size"], L[nrow(L), "train"], L[nrow(L), "cv"], L[nrow(L), "train"] - L[nrow(L), "cv"]))
}



<p class="caption"><b>Figura 5.</b> Curvas de aprendizaje. Izquierda: regresión logística, cuyas curvas de entrenamiento y de CV convergen hasta quedar a aproximadamente un punto de distancia con el tamaño completo —un modelo de baja varianza que ha extraído la mayor parte de lo que contienen estos datos, de modo que más pacientes ayudarían poco. Derecha: 1-NN, cuyo puntaje de entrenamiento se mantiene en 1.000 mientras el puntaje de CV se estanca por debajo —una brecha persistente que más datos reducen solo lentamente.</p>

### 4.3 La descomposición sesgo–varianza

Ambos diagnósticos tienen una misma teoría detrás. Para un problema de regresión \(y = f(x) + \varepsilon\) con varianza del ruido \(\sigma^2\) y un procedimiento que entrega \(\hat{f}_D(x)\) a partir de una muestra de entrenamiento \(D\), el error cuadrático esperado en \(x\), sobre las muestras y sobre el ruido de una nueva observación, se descompone exactamente como

<div class="eq">\[ \mathbb{E}\big[(y - \hat{f}_D(x))^2\big] = \underbrace{\big(f(x) - \mathbb{E}_D[\hat{f}_D(x)]\big)^2}_{\text{bias}^2} + \underbrace{\mathbb{E}_D\big[(\hat{f}_D(x) - \mathbb{E}_D[\hat{f}_D(x)])^2\big]}_{\text{variance}} + \underbrace{\sigma^2}_{\text{irreducible noise}}, \]
<span class="where">donde \(\mathbb{E}_D\) se toma sobre las muestras de entrenamiento. El <strong>sesgo</strong> es el error que proviene de supuestos erróneos del modelo —una recta supuesta donde la realidad es curva—; es grande en los modelos rígidos, y un sesgo alto produce <em>subajuste</em>. La <strong>varianza</strong> es la sensibilidad del ajuste a pequeñas fluctuaciones del conjunto de entrenamiento; es grande en los modelos flexibles, y una varianza alta produce <em>sobreajuste</em>. El término de ruido es el piso por debajo del cual ningún modelo llega. La descomposición proviene del análisis de redes neuronales de Geman, Bienenstock y Doursat de 1992;<sup class="c"><a href="#r15">[15]</a></sup> el capítulo 7 de Hastie, Tibshirani y Friedman la desarrolla para los métodos de este curso.<sup class="c"><a href="#r18">[18]</a></sup></span></div>

Más flexibilidad reduce el sesgo y aumenta la varianza; el error de prueba, su suma más el ruido, tiene forma de U, y su mínimo corresponde a la complejidad adecuada. La Figura 4 era esta forma de U vista a través de la exactitud de CV. La descomposición no puede calcularse en datos reales —\(f\) es desconocida y solo existe una muestra—, pero sí simularse: 300 muestras de entrenamiento de 20 puntos de \(y = \sin(2\pi x) + \varepsilon\), \(\sigma = 0.3\), un polinomio de cada grado ajustado a cada muestra, y sesgo\(^2\) y varianza medidos sobre los 300 ajustes.



In [ ]:
# Simulación sesgo-varianza: 300 muestras de entrenamiento, ajustes polinomiales de grado 0-12 a y = sin(2 pi x) + ruido.
set.seed(0)
f_true <- function(x) sin(2 * pi * x); n_pts <- 20; sigma <- 0.3; R <- 300
x_fixed <- seq(0, 1, length.out = n_pts)                              # diseño fijo: los mismos x en cada muestra
Y <- matrix(f_true(x_fixed), R, n_pts, byrow = TRUE) + matrix(rnorm(R * n_pts, 0, sigma), R, n_pts)
degrees <- 0:12
fitmat <- function(dg) t(sapply(1:R, function(r) if (dg == 0) rep(mean(Y[r, ]), n_pts) else predict(lm(Y[r, ] ~ poly(x_fixed, dg, raw = TRUE)))))
fits <- lapply(degrees, fitmat); names(fits) <- degrees
Yfresh <- matrix(f_true(x_fixed), R, n_pts, byrow = TRUE) + matrix(rnorm(R * n_pts, 0, sigma), R, n_pts)   # muestra ruidosa nueva
decomp <- data.frame(
  degree = degrees,
  `bias^2` = sapply(fits, function(fm) mean((colMeans(fm) - f_true(x_fixed))^2)),
  variance = sapply(fits, function(fm) mean(apply(fm, 2, var))),
  noise = sigma^2, check.names = FALSE)
decomp$`bias^2 + variance + noise` <- decomp$`bias^2` + decomp$variance + decomp$noise
decomp$`empirical test MSE` <- sapply(fits, function(fm) mean((fm - Yfresh)^2))
round(decomp[c(1, 2, 3, 4, 5, 7, 10, 13), ], 4)



La tabla es la descomposición, grado por grado. En el grado 0 (una recta horizontal) el sesgo\(^2\) es grande y la varianza diminuta; en el grado 3 el sesgo\(^2\) casi ha desaparecido, porque una cúbica sigue de cerca un período de una sinusoide; de ahí en adelante la varianza crece con cada coeficiente mientras el sesgo\(^2\) se mantiene cerca de cero. La columna "bias\(^2\) + variance + noise" coincide con el error cuadrático medio frente a observaciones ruidosas nuevas dentro del error de simulación: la identidad se cumple. La figura muestra lo mismo en forma de curvas.



In [ ]:
# Izquierda: 30 de los 300 ajustes para tres grados; su media (gruesa) y la realidad (discontinua). Derecha: la descomposición.
par(mfrow = c(1, 2))
plot(NA, xlim = c(0, 1), ylim = c(-2, 2), xlab = "x", ylab = "y", main = "Rigid fits agree and miss; flexible scatter")
for (dc in list(c(1, GREY), c(3, BLUE), c(12, ORANGE))) { dg <- as.integer(dc[1]); col <- dc[2]
  for (r in 1:30) lines(x_fixed, fits[[as.character(dg)]][r, ], col = adjustcolor(col, 0.30), lwd = 0.6)
  lines(x_fixed, colMeans(fits[[as.character(dg)]]), col = col, lwd = 2.4) }
lines(x_fixed, f_true(x_fixed), lty = 2, lwd = 1.4); points(x_fixed, Y[1, ], pch = 19, cex = 0.6)
legend("topright", bty = "n", cex = 0.72, lwd = 2, col = c(GREY, BLUE, ORANGE, "black"),
       lty = c(1, 1, 1, 2), legend = c("degree 1 (mean)", "degree 3 (mean)", "degree 12 (mean)", "truth sin(2 pi x)"))
plot(decomp$degree, decomp$`bias^2`, type = "o", pch = 19, col = GREY, ylim = c(0, 0.6),
     xlab = "polynomial degree (flexibility)", ylab = "mean squared error", main = "The U-shaped test error")
lines(decomp$degree, decomp$variance, type = "o", pch = 15, col = ORANGE)
lines(decomp$degree, decomp$`bias^2 + variance + noise`, type = "o", pch = 17, col = BLUE)
abline(h = sigma^2, lty = 3)
legend("top", bty = "n", cex = 0.78, pch = c(19, 15, 17, NA), lty = c(1, 1, 1, 3), col = c(GREY, ORANGE, BLUE, "black"),
       legend = c("bias^2", "variance", "total = test MSE", sprintf("noise floor = %.2f", sigma^2)))



<p class="caption"><b>Figura 6.</b> Izquierda: treinta ajustes de cada grado a treinta muestras de entrenamiento. Las rectas (grado 1) coinciden entre sí y todas yerran la sinusoide: sesgo puro. Las curvas de grado 12 siguen la realidad en promedio, pero cada una divaga: varianza pura. Las cúbicas cumplen ambas tareas de forma adecuada. Derecha: el sesgo\(^2\) baja y la varianza sube con el grado; su suma es mínima en un grado bajo y nunca alcanza el piso de ruido \(\sigma^2 = 0.09\).</p>

<div class="box bridge"><h4>Puente hacia las unidades</h4><p>Cada método del curso tiene una perilla sobre esta curva: \(k\) en k-NN y el ancho del kernel de una SVM (Unidad 3), la profundidad de los árboles y el número de árboles (Unidad 4), las capas y la regularización de una red (Unidades 6 y 8), el grado o la penalización de una regresión (Unidad 2). La perilla se ajusta <em>siempre</em> con datos de validación o con validación cruzada, nunca con el puntaje de entrenamiento, que solo premia la flexibilidad.</p></div>

<div class="box"><h4>Ejercicio 3</h4><p>Cambia <code>n_pts</code> de 20 a 100 en la simulación y vuelve a ejecutar ambos bloques de código. La curva de varianza se aplana y el grado óptimo se desplaza hacia la derecha: con más datos, un modelo más flexible se vuelve asequible. Luego fija <code>sigma = 0.6</code>. ¿Qué ocurre con el piso de ruido y con el grado en que el error total es mínimo?</p></div>

### 4.4 La maldición de la dimensionalidad y la razón parámetros/observaciones

La complejidad también crece con el número de características. Cada característica añade un parámetro a un modelo lineal y una dimensión al espacio en que k-NN mide distancias, y en dimensiones altas una muestra fija se vuelve dispersa: cada caso está lejos de todos los demás y los vecinos "más cercanos" ya no están cerca —la <strong>maldición de la dimensionalidad</strong> (curse of dimensionality).<sup class="c"><a href="#r18">[18]</a></sup> La medida práctica es la razón parámetros/observaciones \(p/n\). Con \(p \geq n\), un modelo lineal puede ajustar perfectamente cualquier conjunto de etiquetas, de modo que su puntaje de entrenamiento pierde sentido; una regla práctica para los modelos de predicción clínica exige al menos diez a veinte eventos por parámetro candidato.<sup class="c"><a href="#r27">[27]</a></sup> El bloque de código añade características de ruido puro a las nueve reales.



In [ ]:
# Maldición de la dimensionalidad: características de ruido añadidas a las nueve reales (n = 683 fijo); exactitud de entrenamiento frente a CV.
set.seed(0); cv5 <- strat_kfold(y_pos, 5, 0); rows <- list()
for (p_noise in c(0, 30, 100, 300, 1000)) {
  Xp <- scale(cbind(X, matrix(rnorm(nrow(X) * p_noise), nrow(X), p_noise)))
  dfp <- data.frame(Xp, y = y_pos)
  knn_tr <- mean(knn(Xp, Xp, factor(y_pos), k = 5) == y_pos)
  knn_cv <- mean(sapply(cv5, function(f) mean(knn(Xp[f$train, ], Xp[f$test, ], factor(y_pos[f$train]), k = 5) == y_pos[f$test])))
  fit <- suppressWarnings(glm(y ~ ., dfp, family = binomial)); lr_tr <- mean((predict(fit, type = "response") >= 0.5) == y_pos)
  lr_cv <- mean(sapply(cv5, function(f) { fit <- suppressWarnings(glm(y ~ ., dfp[f$train, ], family = binomial))
    mean((predict(fit, dfp[f$test, ], type = "response") >= 0.5) == y_pos[f$test]) }))
  rows[[length(rows) + 1]] <- data.frame(`noise added` = p_noise, `p/n` = round(ncol(Xp) / nrow(X), 2),
    `5NN train` = knn_tr, `5NN CV` = knn_cv, `logit train` = lr_tr, `logit CV` = lr_cv, check.names = FALSE)
}
curse <- do.call(rbind, rows); rownames(curse) <- NULL; curse[, 3:6] <- round(curse[, 3:6], 3); curse



La información es idéntica en todas las filas —las mismas nueve características reales— y solo se añadieron columnas inútiles. Cuando \(p/n\) supera 1, la exactitud de entrenamiento de la regresión logística llega a 1.000 (ahora puede separar cualquier asignación de etiquetas) mientras su exactitud de CV disminuye; en k-NN ambos puntajes caen, porque las distancias pasan a estar dominadas por el ruido. La selección de características y la regularización (Unidades 2–4) existen para combatir esto, y deben a su vez someterse a validación cruzada (Sección 7.1).

## 5. Métricas para clasificación

### 5.1 La matriz de confusión

Toda métrica de un clasificador binario es función de cuatro recuentos. Fija qué clase es la <strong>positiva</strong> —en medicina, la condición que se busca detectar— y cruza la clase verdadera con la predicha:

| | predicho positivo | predicho negativo | |
|---|---|---|---|
| **realmente positivo** | TP (verdadero positivo) | FN (falso negativo, <em>caso no detectado</em>) | \(P = \mathrm{TP} + \mathrm{FN}\) |
| **realmente negativo** | FP (falso positivo, <em>falsa alarma</em>) | TN (verdadero negativo) | \(N = \mathrm{FP} + \mathrm{TN}\) |

A partir de ellos, las métricas que usa el curso, con los nombres del aprendizaje automático y los clínicos lado a lado (las notas de Altman y Bland en el BMJ siguen siendo la explicación breve más clara del vocabulario clínico<sup class="c"><a href="#r16">[16]</a>,<a href="#r17">[17]</a></sup>):

| Métrica | Fórmula | Nombre clínico | Se lee como |
|---|---|---|---|
| Exactitud (accuracy) | \((\mathrm{TP}+\mathrm{TN})/(P+N)\) | — | fracción de todos los casos clasificados correctamente |
| Sensibilidad = **recall** = TPR | \(\mathrm{TP}/P\) | sensibilidad | de los enfermos, la fracción detectada |
| Especificidad = TNR | \(\mathrm{TN}/N\) | especificidad | de los sanos, la fracción descartada |
| Tasa de falsos positivos (FPR) | \(\mathrm{FP}/N = 1 - \text{specificity}\) | — | de los sanos, la fracción con falsa alarma |
| **Precisión** = PPV | \(\mathrm{TP}/(\mathrm{TP}+\mathrm{FP})\) | PPV | de los predichos positivos, la fracción realmente enferma |
| Valor predictivo negativo (NPV) | \(\mathrm{TN}/(\mathrm{TN}+\mathrm{FN})\) | NPV | de los predichos negativos, la fracción realmente sana |
| Puntaje F1 | \(2\,\mathrm{TP}/(2\,\mathrm{TP}+\mathrm{FP}+\mathrm{FN})\) | — | media armónica de precisión y sensibilidad; ignora los TN |
| Exactitud balanceada | \((\text{sensitivity} + \text{specificity})/2\) | — | la exactitud si ambas clases fueran igual de frecuentes<sup class="c"><a href="#r29">[29]</a></sup> |
| Coeficiente de correlación de Matthews | \(\dfrac{\mathrm{TP}\cdot\mathrm{TN} - \mathrm{FP}\cdot\mathrm{FN}}{\sqrt{(\mathrm{TP}+\mathrm{FP})(\mathrm{TP}+\mathrm{FN})(\mathrm{TN}+\mathrm{FP})(\mathrm{TN}+\mathrm{FN})}}\) | — | correlación entre la clase verdadera y la predicha, de −1 a +1, 0 al azar<sup class="c"><a href="#r7">[7]</a></sup> |

La sensibilidad y la especificidad son propiedades de la <em>prueba</em>, calculadas dentro de los enfermos y dentro de los sanos, por lo que no cambian cuando la enfermedad se vuelve más rara. El PPV y el NPV son propiedades de la prueba <em>en una población</em> y dependen de cuán frecuente sea la enfermedad (Sección 5.3).

### 5.2 Ejemplo resuelto a mano

Una prueba de tamizaje se aplica a 100 pacientes, de los cuales 20 tienen la enfermedad. La prueba marca a 23 pacientes: 18 que tienen la enfermedad y 5 que no. Así, \(\mathrm{TP} = 18\), \(\mathrm{FN} = 2\), \(\mathrm{FP} = 5\), \(\mathrm{TN} = 75\).

- Exactitud \(= (18 + 75)/100 = 0.930\)
- Sensibilidad \(= 18/20 = 0.900\); especificidad \(= 75/80 = 0.9375\)
- PPV \(= 18/23 \approx 0.783\); NPV \(= 75/77 \approx 0.974\)
- F1 \(= 2 \cdot 18/(2 \cdot 18 + 5 + 2) = 36/43 \approx 0.837\)
- Exactitud balanceada \(= (0.900 + 0.9375)/2 \approx 0.919\)
- MCC \(= (18 \cdot 75 - 5 \cdot 2)/\sqrt{23 \cdot 20 \cdot 80 \cdot 77} = 1340/\sqrt{2\,833\,600} \approx 0.796\)

El bloque de código calcula los mismos números a partir de los cuatro recuentos y, como verificación cruzada, reconstruye los 100 pacientes como vectores de etiquetas/predicciones y recupera los recuentos con <code>table()</code>.



In [ ]:
# La matriz de confusión resuelta a mano, como código: métricas a partir de los cuatro recuentos y verificación cruzada con table().
metrics_from_counts <- function(tp, fn, fp, tn) {
  tp <- as.numeric(tp); fn <- as.numeric(fn); fp <- as.numeric(fp); tn <- as.numeric(tn)   # evita el desbordamiento de enteros en el MCC
  P <- tp + fn; N <- fp + tn; sens <- tp / P; spec <- tn / N
  dv <- function(a, b) if (b > 0) a / b else 0                       # una razón con denominador vacío se informa como 0
  mcc_den <- sqrt((tp + fp) * (tp + fn) * (tn + fp) * (tn + fn))
  c(accuracy = (tp + tn) / (P + N), `sensitivity (recall)` = sens, specificity = spec,
    `precision (PPV)` = dv(tp, tp + fp), NPV = dv(tn, tn + fn), F1 = dv(2 * tp, 2 * tp + fp + fn),
    `balanced accuracy` = (sens + spec) / 2, MCC = dv(tp * tn - fp * fn, mcc_den))
}
# reconstruye los 100 pacientes del ejemplo como vectores de etiquetas/predicciones (1 = enfermedad)
yt <- c(rep(1, 20), rep(0, 80)); yp <- c(rep(1, 18), rep(0, 2), rep(1, 5), rep(0, 75))
cm <- table(truth = factor(yt, c(1, 0)), predicted = factor(yp, c(1, 0)))
print(cm)
tp <- cm["1", "1"]; fn <- cm["1", "0"]; fp <- cm["0", "1"]; tn <- cm["0", "0"]
cat(sprintf("\nTP=%d FN=%d FP=%d TN=%d\n", tp, fn, fp, tn))
round(metrics_from_counts(tp, fn, fp, tn), 4)



El vector calculado coincide con el cálculo a mano hasta el último dígito. Dos observaciones prácticas. Una tabla de confusión solo es inequívoca una vez fijados los niveles del factor (aquí <code>factor(.., c(1,0))</code> pone primero la clase positiva, lo que da la disposición epidemiológica [[TP, FN], [FP, TN]]); el orden alfabético/numérico por defecto pondría TN en la esquina superior izquierda, la fuente clásica de sensibilidad y especificidad intercambiadas. Además, toda métrica que depende de la clase tiene una imagen especular: la especificidad es la sensibilidad de la clase negativa, y el NPV su precisión.

<div class="box warn"><h4>¿Qué clase es la positiva?</h4><p>Los datos de Wisconsin de <code>mlbench</code> codifican la etiqueta como un factor con niveles <code>benign</code> y luego <code>malignant</code>, de modo que <code>as.integer(Class)</code> haría benigno = 1 —lo contrario de la convención clínica. Cada bloque de código de este notebook define <code>y_pos = as.integer(Class == "malignant")</code> para que 1 sea la condición que se busca detectar. <em>¿Qué etiqueta es el 1?</em> debería ser la primera línea de cualquier evaluación que escribas.</p></div>



In [ ]:
# Métricas de un clasificador de regresión logística en los datos de cáncer de mama, positivo = maligno, mediante predicciones de CV de 10 particiones.
cvprob_glm <- function(Xm, y, k = 10, seed = 0) {           # P(maligno) out-of-fold para cada caso
  f <- strat_folds(y, k, seed); p <- numeric(length(y)); df <- data.frame(Xm, y = y)
  for (j in 1:k) { tr <- f != j; fit <- suppressWarnings(glm(y ~ ., df[tr, ], family = binomial))
    p[!tr] <- predict(fit, df[!tr, ], type = "response") }
  p
}
prob_cv <- cvprob_glm(as.matrix(Xs), y_pos)                 # reutilizado en las Secciones 5-8
pred_cv <- as.integer(prob_cv >= 0.5)
tp <- sum(pred_cv == 1 & y_pos == 1); fn <- sum(pred_cv == 0 & y_pos == 1)
fp <- sum(pred_cv == 1 & y_pos == 0); tn <- sum(pred_cv == 0 & y_pos == 0)
cm <- matrix(c(tp, fp, fn, tn), 2, 2, dimnames = list(c("truly malignant", "truly benign"), c("pred. malignant", "pred. benign")))
print(cm)
round(metrics_from_counts(tp, fn, fp, tn), 3)



En términos clínicos: de los 239 tumores malignos, unos pocos no se detectan (FN) —cada uno, un diagnóstico tardío— y de los 444 benignos, unos pocos se marcan (FP), cada uno, una biopsia innecesaria. La sensibilidad y la especificidad están ambas en el rango 0.95–0.98 y, como las clases son de tamaño comparable, la exactitud, la exactitud balanceada, el F1 y el MCC coinciden. Esa coincidencia es una propiedad del balance de este conjunto de datos, no de las métricas.

### 5.3 Desbalance: cuándo falla la exactitud y por qué el PPV depende de la prevalencia

La mayoría de los problemas de tamizaje están <strong>desbalanceados</strong>: la clase de interés, la <strong>clase minoritaria</strong>, es rara y la <strong>clase mayoritaria</strong> domina. El grado de desbalance se describe convencionalmente por la proporción de la clase minoritaria:

| Grado de desbalance | Proporción de la clase minoritaria | Ejemplo |
|---|---|---|
| Leve | 20–40 % | tumores malignos en la muestra de Wisconsin (35 %) |
| Moderado | 1–20 % | retinopatía diabética en un programa de tamizaje |
| Extremo | < 1 % | una enfermedad rara, un fraude, una falla crítica de un dispositivo |

El ejemplo de las diapositivas de la clase muestra por qué la exactitud falla aquí. Se aplica un filtro de spam a 10,000 correos electrónicos, de los cuales 100 son spam. El filtro que clasifica <em>todos</em> los correos como "no spam" obtiene \(\mathrm{TN} = 9{,}900\), \(\mathrm{FN} = 100\), \(\mathrm{TP} = 0\), \(\mathrm{FP} = 0\), y por tanto una exactitud de \(9{,}900/10{,}000 = 99\,\%\) sin detectar nada. La simulación reproduce la situación con una lectura médica: 20,000 casos, 1 % de prevalencia, una regresión logística, y el clasificador que predice "sano" para todos.



In [ ]:
# Desbalance: 1 % de prevalencia. Compara un clasificador real con la regla trivial 'todos están sanos'.
gen_imbalanced <- function(n, prev, p = 10, seed = 0, sep = 1.0) {     # gaussianas condicionales a la clase, 5 características informativas
  set.seed(seed); y <- as.integer(runif(n) < prev); Xm <- matrix(rnorm(n * p), n, p)
  Xm[y == 1, 1:5] <- Xm[y == 1, 1:5] + sep; list(X = Xm, y = y)
}
d <- gen_imbalanced(20000, 0.01, sep = 1.0)
fi <- strat_folds(d$y, 2, 0); tr <- fi == 1; dfi <- data.frame(d$X, y = d$y)
yi_te <- d$y[!tr]
fit_i <- suppressWarnings(glm(y ~ ., dfi[tr, ], family = binomial)); prob_imb <- predict(fit_i, dfi[!tr, ], type = "response")
cat(sprintf("prevalence of the positive class: %.3f  (%d positives among %d test cases)\n", mean(d$y), sum(yi_te), length(yi_te)))
res <- list()
for (nm in c("'everyone is healthy'", "logistic regression")) {
  pred <- if (grepl("healthy", nm)) rep(0L, length(yi_te)) else as.integer(prob_imb >= 0.5)
  tp <- sum(pred == 1 & yi_te == 1); fn <- sum(pred == 0 & yi_te == 1); fp <- sum(pred == 1 & yi_te == 0); tn <- sum(pred == 0 & yi_te == 0)
  m <- round(metrics_from_counts(tp, fn, fp, tn), 3); res[[nm]] <- c(m, `TP/FN/FP` = sprintf("%d/%d/%d", tp, fn, fp))
}
as.data.frame(do.call(rbind, res))



La regla trivial alcanza una exactitud cercana a 0.99 sin detectar un solo caso: la sensibilidad, la precisión, el F1 y el MCC son cero, y la exactitud balanceada es exactamente 0.5, el azar. La regresión logística tiene una exactitud muy similar y, sin embargo, detecta una parte de los positivos con alta precisión. Ante el desbalance, la exactitud mide la prevalencia, no el clasificador; las métricas informativas miran dentro de cada clase (sensibilidad, especificidad, exactitud balanceada, MCC) o dentro del grupo de predichos positivos (precisión, F1). Chicco y Jurman defienden el MCC como el resumen único más informativo, porque solo es alto cuando las cuatro celdas son buenas;<sup class="c"><a href="#r8">[8]</a></sup> el F1, que ignora los verdaderos negativos, es la convención de la recuperación de información, donde los negativos son irrelevantes e incontables —algo que rara vez ocurre en medicina.

**Entrenar con datos desbalanceados: downsampling y upweighting.** Prueba primero con los datos originales —la regresión logística maneja correctamente un desbalance moderado cuando las métricas se eligen bien— e interviene solo cuando la clase minoritaria es tan rara que el modelo apenas la ve. La intervención estándar es aplicar <em>downsampling</em> a la clase mayoritaria y <em>upweighting</em> a los casos retenidos en el mismo factor, de modo que las probabilidades predichas sigan significando lo que dicen. Aplicar downsampling sin upweighting, o usar pesos inversos a la frecuencia ("balanceados"), distorsiona deliberadamente esas proporciones: ambas opciones aumentan la sensibilidad y ambas rompen la calibración (Sección 5.6). El bloque de código demuestra las opciones en un problema 1 : 200.



In [ ]:
# Downsampling de la clase mayoritaria por 10 y upweighting por 10 (weights) en un problema 1:200; pesos inversos a la frecuencia como contraste.
d <- gen_imbalanced(60000, 0.005, sep = 1.0, seed = 0)
fi <- strat_folds(d$y, 2, 0); tr <- fi == 1; df <- data.frame(d$X, y = d$y)
dtr <- df[tr, ]; dte <- df[!tr, ]; yte <- d$y[!tr]
pos <- which(dtr$y == 1); neg <- which(dtr$y == 0); set.seed(0); keep <- c(pos, sample(neg, length(neg) %/% 10))
w_up <- ifelse(dtr$y[keep] == 1, 1, 10)                               # upweighting de los casos mayoritarios retenidos en el mismo factor
w_bal <- ifelse(dtr$y == 1, 0.5 / mean(dtr$y), 0.5 / (1 - mean(dtr$y)))
report <- function(fit, ntr, ratio) { pr <- predict(fit, dte, type = "response"); pred <- as.integer(pr >= 0.5)
  tp <- sum(pred == 1 & yte == 1); fn <- sum(pred == 0 & yte == 1); fp <- sum(pred == 1 & yte == 0); tn <- sum(pred == 0 & yte == 0)
  round(c(`training cases` = ntr, ratio = ratio, sensitivity = tp / (tp + fn), specificity = tn / (tn + fp),
          PPV = tp / max(tp + fp, 1), AUC = as.numeric(auc(yte, pr, quiet = TRUE)), Brier = mean((pr - yte)^2),
          `mean pred. P` = mean(pr), `obs. prevalence` = mean(yte)), 4) }
tab <- rbind(
  "original data (1:200)"    = report(suppressWarnings(glm(y ~ ., dtr, family = binomial)), nrow(dtr), 200),
  "downsampled only (1:20)"  = report(suppressWarnings(glm(y ~ ., dtr[keep, ], family = binomial)), length(keep), 20),
  "downsampled + upweighted" = report(suppressWarnings(glm(y ~ ., dtr[keep, ], family = binomial, weights = w_up)), length(keep), 20),
  "inverse-frequency weights" = report(suppressWarnings(glm(y ~ ., dtr, family = binomial, weights = w_bal)), nrow(dtr), 200))
as.data.frame(tab)



Lee primero las dos últimas columnas. Con los datos originales, la probabilidad predicha media es igual a la prevalencia observada (aproximadamente 0.005). El downsampling por sí solo la multiplica varias veces —el riesgo de cada paciente queda sobrestimado— mientras el AUC apenas se mueve; añadir el peso de 10 la devuelve a la prevalencia con la misma discriminación y una décima parte de los casos mayoritarios. La ponderación inversa a la frecuencia eleva drásticamente la sensibilidad, al precio de una especificidad y un PPV mucho más bajos y un puntaje de Brier mucho peor: ha convertido un modelo de probabilidad en un modelo de ordenamiento (ranking), lo que solo está bien si esa era la intención.

**El PPV depende de la prevalencia.** La sensibilidad y la especificidad las fija la prueba, pero la probabilidad de que un paciente marcado esté realmente enfermo no. Escribe \(\pi\) para la prevalencia, \(\mathrm{Se}\) para la sensibilidad y \(\mathrm{Sp}\) para la especificidad. El teorema de Bayes da

<div class="eq">\[ \mathrm{PPV} = P(\text{disease} \mid \text{test}+) = \frac{\mathrm{Se}\,\pi}{\mathrm{Se}\,\pi + (1 - \mathrm{Sp})(1 - \pi)}, \qquad \mathrm{NPV} = \frac{\mathrm{Sp}\,(1-\pi)}{\mathrm{Sp}\,(1-\pi) + (1 - \mathrm{Se})\,\pi}. \]
<span class="where">El numerador es la tasa de verdaderos positivos en la población; el segundo término del denominador es la tasa de falsas alarmas, proporcional a la fracción <em>sana</em> \(1 - \pi\). Cuando la enfermedad es rara, los sanos son muchos, e incluso una tasa pequeña de falsos positivos produce más falsas alarmas que detecciones.<sup class="c"><a href="#r17">[17]</a></sup> La Unidad 3 deduce el teorema de Bayes; aquí se aplica.</span></div>



In [ ]:
# PPV en función de la prevalencia para una prueba fija (teorema de Bayes), con tres niveles de especificidad.
ppv <- function(prev, se, sp) se * prev / (se * prev + (1 - sp) * (1 - prev))
prev <- 10^seq(-3, 0, length.out = 300)                              # prevalencia de 0.1% a 100%
plot(prev * 100, ppv(prev, 0.90, 0.90), type = "l", log = "x", col = GREY, lwd = 2, ylim = c(0, 1),
     xlab = "prevalence of the condition (%)", ylab = "PPV: P(disease | test positive)", main = "The same test, in populations of different prevalence")
lines(prev * 100, ppv(prev, 0.90, 0.99), col = BLUE, lwd = 2); lines(prev * 100, ppv(prev, 0.90, 0.999), col = GREEN, lwd = 2)
for (p0 in c(1, 10, 50)) abline(v = p0, col = ORANGE, lty = 3)
legend("topleft", bty = "n", cex = 0.8, lwd = 2, col = c(GREY, BLUE, GREEN), legend = c("sens 0.90, spec 0.90", "sens 0.90, spec 0.99", "sens 0.90, spec 0.999"))
tab <- sapply(c(0.01, 0.10, 0.50), function(p0) sapply(c(0.90, 0.99, 0.999), function(sp) ppv(p0, 0.90, sp)))
dimnames(tab) <- list(c("specificity 0.90", "specificity 0.99", "specificity 0.999"), c("prevalence 1%", "prevalence 10%", "prevalence 50%"))
round(tab, 3)



<p class="caption"><b>Figura 7.</b> PPV en función de la prevalencia (eje logarítmico) para una sensibilidad de 0.90 y tres especificidades; líneas punteadas en 1 %, 10 % y 50 %. Una prueba con 90 % de especificidad —respetable donde la mitad de los pacientes derivados está enferma— tiene un PPV de aproximadamente 8 % al tamizar una población con 1 % de prevalencia: once de cada doce positivos son falsas alarmas.</p>

La lección se generaliza: un modelo desarrollado donde el 40 % de los pacientes presentaba el desenlace tendrá un PPV mucho menor en atención primaria, donde lo presenta el 4 %, sin ningún cambio en el modelo. La sensibilidad y la especificidad se trasladan entre contextos; los valores predictivos no, y un informe que entrega precisión y sensibilidad sin la prevalencia de su conjunto de prueba ha omitido la información necesaria para usarlo.

### 5.4 De los puntajes a las curvas: ROC

La mayoría de los clasificadores entrega un <strong>puntaje</strong> —una probabilidad, una distancia, un recuento de votos— y la clase se obtiene comparándolo con un <strong>umbral</strong>; 0.5 es una convención, no una ley. Un umbral más alto declara positivos menos casos, de modo que la precisión tiende a subir y la sensibilidad baja; uno más bajo hace lo contrario. La precisión y la sensibilidad se mueven en direcciones opuestas al cambiar el umbral; por eso ninguna se informa sin la otra, y por eso el F1 —su media armónica con igual peso— es un compromiso y no una solución. La tasa de falsos positivos (FP/N) tiene la misma dependencia y solo tiene sentido cuando hay suficientes verdaderos negativos para estimarla, una de las razones por las que se prefiere la curva de precisión–sensibilidad cuando los negativos son escasos.

La curva <strong>ROC</strong> (receiver operating characteristic, característica operativa del receptor) muestra toda la familia de clasificadores que se obtiene al recorrer el umbral: tasa de verdaderos positivos (sensibilidad) frente a tasa de falsos positivos (1 − especificidad). El área bajo ella, el <strong>AUC</strong>, resume la calidad del ordenamiento; Hanley y McNeil mostraron que es igual a la probabilidad de que un positivo elegido al azar reciba un puntaje más alto que un negativo elegido al azar.<sup class="c"><a href="#r4">[4]</a></sup> Un AUC de 1.0 es un ordenamiento perfecto; 0.5 es aleatorio. El tutorial de Fawcett es la referencia estándar sobre su construcción y sus trampas.<sup class="c"><a href="#r5">[5]</a></sup>

**A mano.** Seis pacientes, tres enfermos (D) y tres sanos (H), reciben los puntajes D 0.9, D 0.7, H 0.6, D 0.4, H 0.3, H 0.2. Ordénalos por puntaje y baja el umbral un caso a la vez; cada paso mueve el punto de operación hacia arriba (D) o hacia la derecha (H) en un tercio:

| umbral justo por debajo de | caso | TP | FP | TPR = TP/3 | FPR = FP/3 |
|---|---|---|---|---|---|
| (ninguno positivo) | — | 0 | 0 | 0 | 0 |
| 0.9 | D | 1 | 0 | 1/3 | 0 |
| 0.7 | D | 2 | 0 | 2/3 | 0 |
| 0.6 | H | 2 | 1 | 2/3 | 1/3 |
| 0.4 | D | 3 | 1 | 1 | 1/3 |
| 0.3 | H | 3 | 2 | 1 | 2/3 |
| 0.2 | H | 3 | 3 | 1 | 1 |

El área bajo la escalera es \(\tfrac{1}{3}\cdot\tfrac{2}{3} + \tfrac{2}{3}\cdot 1 = \tfrac{8}{9} \approx 0.889\). La interpretación por pares coincide: de los \(3 \times 3 = 9\) pares (D, H), los D con puntaje 0.9 y 0.7 superan cada uno a los tres H (6 pares) y el D con puntaje 0.4 supera a dos (2 pares); \(8/9\). El bloque de código confirma ambos resultados y los contrasta con el paquete <code>pROC</code>,<sup class="c"><a href="#r33">[33]</a></sup> y luego los aplica a las probabilidades de validación cruzada de la Sección 5.2.



In [ ]:
# Curva ROC y AUC a mano (ordenamiento y conteo acumulado), contrastados con pROC.
roc_by_hand <- function(y_true, score) {                             # vectores FPR, TPR y el AUC por trapecios
  ord <- order(-score); yt <- y_true[ord]
  tpr <- c(0, cumsum(yt == 1)) / sum(yt == 1); fpr <- c(0, cumsum(yt == 0)) / sum(yt == 0)
  list(fpr = fpr, tpr = tpr, auc = sum(diff(fpr) * (head(tpr, -1) + tail(tpr, -1)) / 2))
}
auc_by_pairs <- function(y_true, score) {                            # fracción de pares (pos, neg) en que el positivo tiene mayor puntaje
  d <- outer(score[y_true == 1], score[y_true == 0], "-"); (sum(d > 0) + 0.5 * sum(d == 0)) / length(d)
}
toy_y <- c(1, 1, 0, 1, 0, 0); toy_s <- c(0.9, 0.7, 0.6, 0.4, 0.3, 0.2); rt <- roc_by_hand(toy_y, toy_s)
cat("toy example: FPR", round(rt$fpr, 3), "\n             TPR", round(rt$tpr, 3), "\n")
cat(sprintf("toy AUC: trapezoid %.4f | pairwise %.4f | pROC %.4f  (8/9 = %.4f)\n",
            rt$auc, auc_by_pairs(toy_y, toy_s), as.numeric(auc(toy_y, toy_s, quiet = TRUE, direction = "<")), 8/9))
rh <- roc_by_hand(y_pos, prob_cv)
cat(sprintf("breast cancer (CV probabilities): AUC by hand %.4f | pairwise %.4f | pROC %.4f\n",
            rh$auc, auc_by_pairs(y_pos, prob_cv), as.numeric(auc(y_pos, prob_cv, quiet = TRUE))))



Los tres cálculos coinciden en el ejemplo de juguete (8/9) y en el real, donde el AUC de las probabilidades de la regresión logística con validación cruzada es de aproximadamente 0.99. El bloque de código siguiente dibuja la curva ROC y, a su lado, la curva de <strong>precisión–sensibilidad</strong> (precision–recall, PR) para el problema balanceado de cáncer de mama y para el problema con 1 % de la Sección 5.3, porque la comparación es justamente lo importante.



In [ ]:
# Curvas ROC y de precisión-sensibilidad para un problema balanceado (cáncer de mama) y uno desbalanceado (1% de prevalencia).
pr_by_hand <- function(y_true, score) { ord <- order(-score); yt <- y_true[ord]
  tp <- cumsum(yt == 1); fp <- cumsum(yt == 0); list(recall = tp / sum(yt == 1), precision = tp / (tp + fp)) }
cases <- list("breast cancer (35% malignant)" = list(y_pos, prob_cv, BLUE), "simulated screening (1% positive)" = list(yi_te, prob_imb, ORANGE))
par(mfrow = c(1, 2))
plot(NA, xlim = c(0, 1), ylim = c(0, 1.02), xlab = "false-positive rate (1 - specificity)", ylab = "true-positive rate (sensitivity)", main = "ROC curves (dot = threshold 0.5)")
abline(0, 1, lty = 2); for (nm in names(cases)) { cs <- cases[[nm]]; r <- roc_by_hand(cs[[1]], cs[[2]])
  lines(r$fpr, r$tpr, col = cs[[3]], lwd = 2)
  i0 <- which.min(abs(sort(cs[[2]], decreasing = TRUE) - 0.5)); points(r$fpr[i0 + 1], r$tpr[i0 + 1], pch = 21, bg = cs[[3]], cex = 1.2) }
legend("bottomright", bty = "n", cex = 0.75, lwd = 2, col = c(BLUE, ORANGE, "black"), lty = c(1, 1, 2),
       legend = c(sprintf("breast cancer: AUC = %.3f", roc_by_hand(y_pos, prob_cv)$auc), sprintf("screening 1%%: AUC = %.3f", roc_by_hand(yi_te, prob_imb)$auc), "chance (AUC = 0.5)"))
plot(NA, xlim = c(0, 1), ylim = c(0, 1.02), xlab = "recall (sensitivity)", ylab = "precision (PPV)", main = "Precision-recall curves (dotted = prevalence)")
for (nm in names(cases)) { cs <- cases[[nm]]; pr <- pr_by_hand(cs[[1]], cs[[2]]); lines(pr$recall, pr$precision, col = cs[[3]], lwd = 2); abline(h = mean(cs[[1]]), col = cs[[3]], lty = 3) }
legend("bottomleft", bty = "n", cex = 0.75, lwd = 2, col = c(BLUE, ORANGE), legend = c("breast cancer", "screening 1%"))



<p class="caption"><b>Figura 8.</b> Izquierda: curvas ROC de los dos problemas; ambos AUC son altos y ambas curvas se pegan a la esquina superior izquierda. Derecha: curvas PR de los mismos modelos. En el problema balanceado la curva PR es tan favorecedora como la curva ROC; en el problema con 1 % cae rápidamente, y su línea base (punteada), la precisión de un clasificador aleatorio, es la prevalencia, 0.01. Los puntos marcan el umbral 0.5.</p>

La curva ROC es invariante a la prevalencia porque ambos ejes son tasas dentro de cada clase: esa es su fortaleza para comparar pruebas entre poblaciones y su debilidad en problemas de enfermedades raras, donde una tasa de falsos positivos de 2 % parece despreciable pero, aplicada a 9,900 personas sanas, produce unas 200 falsas alarmas por cada 50 detecciones. La curva PR lo deja en evidencia, y por eso Saito y Rehmsmeier la recomiendan para datos desbalanceados;<sup class="c"><a href="#r6">[6]</a></sup> Davis y Goadrich mostraron que una curva domina en el espacio ROC si y solo si domina en el espacio PR, de modo que ambas nunca discrepan sobre <em>cuál</em> modelo es mejor, solo sobre cuánto mejor parece.<sup class="c"><a href="#r30">[30]</a></sup>

### 5.5 Elegir un umbral según el costo

Una curva ROC es un menú; desplegar un clasificador significa elegir un punto en ella, y el punto correcto depende de cuánto cuestan los errores. Si un falso negativo cuesta \(c_{\mathrm{FN}}\) y un falso positivo \(c_{\mathrm{FP}}\), un paciente con probabilidad calibrada \(p\) debe declararse positivo cuando el costo esperado de declararlo negativo supera al de declararlo positivo, \(p\,c_{\mathrm{FN}} > (1-p)\,c_{\mathrm{FP}}\), es decir, cuando

<div class="eq">\[ p > t^{*} = \frac{c_{\mathrm{FP}}}{c_{\mathrm{FP}} + c_{\mathrm{FN}}}. \]
<span class="where">Si se considera que un cáncer no detectado cuesta diez veces más que una biopsia innecesaria, \(t^{*} = 1/11 \approx 0.09\), muy por debajo de 0.5. El <em>análisis de curvas de decisión</em> (decision curve analysis) de Vickers y Elkin incorpora este razonamiento en un método gráfico estándar.<sup class="c"><a href="#r31">[31]</a></sup> El bloque de código calcula el costo total por cada 100 pacientes para cada umbral y compara el mínimo empírico con \(t^{*}\).</span></div>



In [ ]:
# Costo esperado por cada 100 pacientes en función del umbral de decisión, para tres razones de costo.
thresholds <- seq(0.01, 0.99, length.out = 197)
cost_curve <- function(y_true, prob, c_fn, c_fp) sapply(thresholds, function(t) {
  pred <- as.integer(prob >= t); (sum(pred == 0 & y_true == 1) * c_fn + sum(pred == 1 & y_true == 0) * c_fp) / length(y_true) * 100 })
plot(NA, xlim = c(0, 1), ylim = c(0, 20), xlab = "decision threshold on P(malignant)", ylab = "total cost per 100 patients (biopsy = 1)",
     main = "Choosing the threshold by the cost of the two errors (dotted = t*)")
for (cc in list(c(1, 1, GREY), c(10, 1, BLUE), c(30, 1, ORANGE))) {
  c_fn <- as.numeric(cc[1]); c_fp <- as.numeric(cc[2]); col <- cc[3]; curve_v <- cost_curve(y_pos, prob_cv, c_fn, c_fp)
  t_emp <- thresholds[which.min(curve_v)]; t_th <- c_fp / (c_fp + c_fn)
  lines(thresholds, curve_v, col = col, lwd = 2); abline(v = t_th, col = col, lty = 3)
  cat(sprintf("cost ratio %2d:1  cost at t=0.5: %5.1f  at empirical optimum t=%.2f: %5.1f  at t*=%.3f: %5.1f\n",
              c_fn, curve_v[which.min(abs(thresholds - 0.5))], t_emp, min(curve_v), t_th, curve_v[which.min(abs(thresholds - t_th))]))
}
legend("top", bty = "n", cex = 0.78, lwd = 2, col = c(GREY, BLUE, ORANGE), legend = c("missed = 1 x biopsy", "missed = 10 x biopsy", "missed = 30 x biopsy"))



<p class="caption"><b>Figura 9.</b> Costo total por cada 100 pacientes en función del umbral para tres razones de costo, a partir de las probabilidades de validación cruzada. Con costos asimétricos, el óptimo empírico sigue de cerca el \(t^{*}\) teórico (punteado) —aproximadamente 0.08 frente a \(t^{*}=0.09\) para una razón 10:1, 0.03 frente a 0.03 para 30:1— y el costo de quedarse en 0.5 crece bruscamente a medida que un cáncer no detectado se vuelve más caro. Con costos iguales, la curva es poco profunda y casi plana, de modo que su mínimo queda mal determinado (aquí cerca de 0.12, pero apenas más barato que 0.5): cuando los dos errores cuestan lo mismo, el umbral casi no importa.</p>

### 5.6 Calibración: una probabilidad debe significar lo que dice

La regla del umbral supone que una probabilidad predicha de 0.3 significa que aproximadamente el 30 % de esos pacientes tiene la enfermedad. Esa propiedad es la <strong>calibración</strong>, y es distinta de la discriminación: un modelo puede ordenar perfectamente a los pacientes (AUC = 1) mientras sus probabilidades están todas comprimidas en [0.45, 0.55] o todas empujadas a 0 y 1. Van Calster y colaboradores llaman a la calibración "el talón de Aquiles de la analítica predictiva", porque rara vez se evalúa y la rompen habitualmente el dataset shift, la reponderación de clases y el sobreajuste;<sup class="c"><a href="#r10">[10]</a></sup> Steyerberg y Vergouwe la sitúan, junto con la discriminación y la utilidad clínica, entre las tres cosas que un modelo de predicción debe demostrar.<sup class="c"><a href="#r14">[14]</a></sup>

Dos herramientas la evalúan. El <strong>diagrama de confiabilidad</strong> (reliability diagram) agrupa los casos en intervalos según la probabilidad predicha y grafica, para cada intervalo, la fracción observada de positivos frente a la predicción media; un modelo calibrado se ubica sobre la diagonal. El <strong>puntaje de Brier</strong> es la diferencia cuadrática media entre la probabilidad predicha y el desenlace,

<div class="eq">\[ \mathrm{BS} = \frac{1}{n} \sum_{i=1}^{n} (p_i - y_i)^2, \]
<span class="where">introducido por Brier en 1950 para los pronósticos meteorológicos;<sup class="c"><a href="#r9">[9]</a></sup> 0 es perfecto, 0.25 corresponde a predecir siempre 0.5, y predecir la prevalencia \(\pi\) para todos obtiene \(\pi(1-\pi)\). Premia a la vez la discriminación y la calibración. El bloque de código compara tres clasificadores del curso usando probabilidades de validación cruzada.</span></div>



In [ ]:
# Diagramas de confiabilidad y puntajes de Brier de tres clasificadores (probabilidades de validación cruzada, positivo = maligno).
cvprob_knn <- function(Xm, y, k = 15, folds = 10, seed = 0) { f <- strat_folds(y, folds, seed); p <- numeric(length(y))
  for (j in 1:folds) { tr <- f != j; pr <- knn(Xm[tr, ], Xm[!tr, ], factor(y[tr]), k = k, prob = TRUE)
    w <- attr(pr, "prob"); p[!tr] <- ifelse(pr == 1, w, 1 - w) }; p }
cvprob_nb <- function(Xm, y, folds = 10, seed = 0) { f <- strat_folds(y, folds, seed); p <- numeric(length(y)); df <- data.frame(Xm, yf = factor(y))
  for (j in 1:folds) { tr <- f != j; m <- naiveBayes(yf ~ ., df[tr, ]); p[!tr] <- predict(m, df[!tr, ], type = "raw")[, 2] }; p }
probs <- list("logistic regression" = prob_cv, "15-nearest neighbours" = cvprob_knn(Xs, y_pos), "Gaussian naive Bayes (Unit 3)" = cvprob_nb(Xs, y_pos))
reldiag <- function(p, y, nb = 8) { br <- unique(quantile(p, seq(0, 1, length.out = nb + 1))); br[1] <- -Inf; br[length(br)] <- Inf
  b <- cut(p, br, labels = FALSE); data.frame(mp = tapply(p, b, mean), fp = tapply(y, b, mean)) }
plot(NA, xlim = c(0, 1), ylim = c(0, 1), xlab = "mean predicted probability of malignancy (8 quantile bins)", ylab = "observed fraction malignant", main = "Reliability diagram")
abline(0, 1, lty = 2); cols <- c(BLUE, GREEN, ORANGE); rows <- list()
for (i in seq_along(probs)) { rd <- reldiag(probs[[i]], y_pos); lines(rd$mp, rd$fp, type = "o", pch = 19, col = cols[i])
  rows[[names(probs)[i]]] <- round(c(`Brier score` = mean((probs[[i]] - y_pos)^2), AUC = as.numeric(auc(y_pos, probs[[i]], quiet = TRUE)),
    `mean pred. P` = mean(probs[[i]]), `share P outside (.02,.98)` = mean(probs[[i]] < 0.02 | probs[[i]] > 0.98)), 3) }
legend("topleft", bty = "n", cex = 0.85, lwd = 2, pch = 19, col = cols, legend = names(probs))
as.data.frame(do.call(rbind, rows))



<p class="caption"><b>Figura 10.</b> Diagrama de confiabilidad de tres clasificadores. La regresión logística, ajustada maximizando la verosimilitud de las etiquetas, sigue de cerca la diagonal. Bayes ingenuo tiene un AUC comparable pero probabilidades extremas —la mayoría por debajo de 0.02 o por encima de 0.98— porque su supuesto de independencia multiplica evidencia que en realidad es redundante; está <em>sobreconfiado</em>, y su puntaje de Brier es notablemente peor. Las fracciones de votos de 15-NN quedan en un punto intermedio.</p>

La tabla separa las dos cualidades: los AUC están a pocas centésimas unos de otros; los puntajes de Brier, no. Clínicamente, la sobreconfianza no es cosmética: un modelo que dice "2 % de riesgo" cuando el riesgo real entre esos pacientes es 15 % se usará para negar una biopsia justificada. Los modelos cuyas probabilidades no están calibradas por construcción —Bayes ingenuo, SVM, la mayoría de las redes— se <em>recalibran</em> con datos de validación (escalado de Platt, regresión isotónica), y la razón observado/esperado (desenlace medio sobre probabilidad predicha media, 1.0 cuando hay calibration-in-the-large) se informa junto al AUC.

<div class="box"><h4>Ejercicio 4</h4><p>Recalibra las probabilidades de Bayes ingenuo ajustando una recalibración isotónica o logística (<code>glm(y ~ p, family = binomial)</code> sobre probabilidades out-of-fold, la forma en R del escalado de Platt) y agrega la curva recalibrada. Debería desplazarse hacia la diagonal y su puntaje de Brier bajar, mientras su AUC apenas cambia. ¿Por qué la recalibración puede mejorar el puntaje de Brier pero no el AUC? (¿Frente a qué operación es invariante la curva ROC?)</p></div>

## 6. Métricas para regresión

Cuando la variable objetivo es un número —progresión de una enfermedad, glucemia, duración de la estadía hospitalaria, el precio de una vivienda—, los errores son los residuos \(e_i = y_i - \hat{y}_i\) y los resúmenes estándar son

<div class="eq">\[ \mathrm{MSE} = \frac{1}{n}\sum e_i^2, \qquad \mathrm{RMSE} = \sqrt{\mathrm{MSE}}, \qquad \mathrm{MAE} = \frac{1}{n}\sum |e_i|, \qquad R^2 = 1 - \frac{\sum (y_i - \hat{y}_i)^2}{\sum (y_i - \bar{y})^2}. \]
<span class="where">El MSE es lo que minimizan los mínimos cuadrados (Unidad 2) y lo que descompone la Sección 4.3; el RMSE recupera las unidades de la variable objetivo; el MAE, en las mismas unidades, es menos sensible a unos pocos errores grandes y es el que un lector puede interpretar directamente. \(R^2\) compara el error cuadrático del modelo con el del predictor más simple, la media de \(y\): 1 para un ajuste perfecto, 0 para un modelo que no es mejor que la media.</span></div>

En la muestra de entrenamiento, el \(R^2\) de un ajuste por mínimos cuadrados está en \([0, 1]\), porque el ajuste siempre puede hacerlo al menos tan bien como la media. En una muestra de <em>prueba</em> nada lo garantiza: un modelo que predice peor que la media de prueba tiene un <strong>\(R^2\) negativo</strong>. Esto no es un error de programación; es el informe correcto de que el modelo es peor que inútil en esos datos. La edición en Python usó los datos de progresión de diabetes de Efron y colaboradores;<sup class="c"><a href="#r21">[21]</a></sup> la edición en R usa los datos de viviendas de Boston de Harrison y Rubinfeld (506 localidades, 13 predictores, el valor mediano de las viviendas como variable objetivo),<sup class="c"><a href="#r36">[36]</a></sup> que provee el paquete <code>mlbench</code>. Los bloques de código muestran el caso sano y el patológico.



In [ ]:
# Métricas de regresión a mano (a partir de los residuos) y el R^2 de entrenamiento que informa summary() de R, con los datos de Boston.
data(BostonHousing, package = "mlbench"); bh <- BostonHousing; bh$chas <- as.numeric(as.character(bh$chas))
set.seed(0); bi <- sample(nrow(bh), round(0.7 * nrow(bh)))
lin <- lm(medv ~ ., bh[bi, ]); yd_te <- bh$medv[-bi]; pe <- predict(lin, bh[-bi, ]); e <- yd_te - pe
by_hand <- c(MSE = mean(e^2), RMSE = sqrt(mean(e^2)), MAE = mean(abs(e)), `test R2` = 1 - sum(e^2) / sum((yd_te - mean(yd_te))^2))
cat(sprintf("target medv: mean %.1f, SD %.1f on the %d test towns | training R2 (summary) = %.3f\n",
            mean(yd_te), sd(yd_te), length(yd_te), summary(lin)$r.squared))
round(by_hand, 3)



El RMSE supera al MAE, como ocurre siempre que unos pocos residuos son grandes; ambos deben leerse frente a la dispersión de la variable objetivo (SD de aproximadamente 9): el modelo elimina la mayor parte de la variación, pero no toda, que es lo que dice en términos relativos el \(R^2\) de prueba. Aquí el \(R^2\) de entrenamiento (aproximadamente 0.74) y el \(R^2\) de prueba (aproximadamente 0.74) son cercanos, porque la regresión lineal es un modelo de baja varianza cuyo optimismo es pequeño; la brecha de la Sección 1.2 aparece solo con un ajuste más flexible, que aporta el bloque de código siguiente. Ahora, el caso patológico.



In [ ]:
# R^2 de prueba negativo: regresión polinomial multivariada de grado creciente sobre la misma división.
preds <- setdiff(names(bh), "medv"); rows <- list()
for (dg in 1:3) {
  form <- as.formula(paste0("medv ~ polym(", paste(preds, collapse = ", "), ", degree = ", dg, ", raw = TRUE)"))
  fit <- suppressWarnings(lm(form, bh[bi, ])); e <- yd_te - predict(fit, bh[-bi, ])
  rows[[dg]] <- data.frame(degree = dg, `no. coefficients` = length(coef(fit)), `train R2` = summary(fit)$r.squared,
                           `test R2` = 1 - sum(e^2, na.rm = TRUE) / sum((yd_te - mean(yd_te))^2),
                           `test RMSE` = sqrt(mean(e^2, na.rm = TRUE)), check.names = FALSE)
}
rmse_mean <- sqrt(mean((yd_te - mean(bh$medv[bi]))^2))
cat(sprintf("RMSE of predicting the training mean for every test town: %.1f  (R2 = %.4f)\n",
            rmse_mean, 1 - sum((yd_te - mean(bh$medv[bi]))^2) / sum((yd_te - mean(yd_te))^2)))
res <- do.call(rbind, rows); rownames(res) <- NULL; res[, 3:5] <- signif(res[, 3:5], 3); res



Un polinomio multivariado de grado 3 tiene cientos de coeficientes ajustados a unas 350 localidades. Su \(R^2\) de entrenamiento sube hasta 1.000 y su \(R^2\) de prueba es un número negativo grande: sus predicciones son mucho peores que asignar a cada localidad el valor promedio. Es la Sección 4 vestida de regresión. Observa también que el predictor trivial de la media obtiene un \(R^2\) prácticamente nulo en el conjunto de prueba (difiere de cero exacto solo porque la media de entrenamiento no es la media de prueba).

<div class="box key"><h4>Cómo informar un modelo de regresión</h4><p>Informa el RMSE o el MAE en las unidades de la variable objetivo, con la SD de la variable objetivo al lado, y el \(R^2\) de prueba. Nunca \(R^2\) solo: 0.9 para una cantidad que varía mil veces puede ocultar errores clínicamente inútiles, y 0.3 puede ser excelente para un desenlace ruidoso.</p></div>

## 7. Fuga de datos

La <strong>fuga de datos</strong> (leakage) es el uso, durante el entrenamiento, de información que no estará disponible cuando el modelo se despliegue —información que se filtra desde el conjunto de prueba, o desde el futuro, hacia el modelo. Produce evaluaciones honestas en la forma (se reservó un conjunto de prueba, se ejecutó una validación cruzada) y erróneas en el fondo. Kaufman y colaboradores dieron la primera descripción sistemática;<sup class="c"><a href="#r11">[11]</a></sup> Kapoor y Narayanan revisaron 294 artículos de 17 campos cuyos resultados se descubrió después que estaban inflados por fuga de datos, y propusieron una taxonomía y una lista de verificación.<sup class="c"><a href="#r12">[12]</a></sup> Esta sección demuestra las tres formas que explican la mayoría de los casos, cada una con el número inflado junto al honesto.

### 7.1 Preprocesar antes de dividir

Todo paso que <em>aprende de los datos</em> —selección de características, estandarización, imputación, reducción de dimensionalidad— debe ajustarse solo en la partición de entrenamiento. Seleccionar características en el conjunto de datos completo y luego aplicar validación cruzada al clasificador es la versión más dañina, porque la selección ya vio las etiquetas de prueba; Ambroise y McLachlan mostraron en 2002 que los clasificadores de expresión génica evaluados de esta forma informaban un error cercano a cero en datos sin señal.<sup class="c"><a href="#r24">[24]</a></sup> El remedio es poner todo paso aprendido <em>dentro</em> del ciclo de validación cruzada, reajustándolo en cada partición de entrenamiento. El bloque de código muestra la fuga en ruido puro (1,000 características, 200 casos, etiquetas aleatorias) y, como contraste, la fuga por escalamiento en los datos de cáncer de mama.



In [ ]:
# Fuga 1: selección de características (y, como contraste, escalamiento) realizada antes de dividir frente a dentro del ciclo de particiones.
select_k <- function(Xm, y, k) order(-abs(apply(Xm, 2, function(col) cor(col, y))))[1:k]   # las k mejores según |correlación| con la etiqueta
set.seed(0); leaky <- honest <- numeric(5)
for (rep in 1:5) {                                                   # 5 conjuntos de datos de ruido independientes
  Xn <- matrix(rnorm(200 * 1000), 200, 1000); yn <- sample(0:1, 200, replace = TRUE); f <- strat_folds(yn, 5, 0)
  sel <- select_k(Xn, yn, 20); Xsel <- Xn[, sel]                     # INCORRECTO: la selección ha visto todas las etiquetas, incluidas las de las particiones de prueba
  leaky[rep] <- mean(sapply(1:5, function(j) { tr <- f != j; mean(knn(Xsel[tr, ], Xsel[!tr, ], factor(yn[tr]), 5) == yn[!tr]) }))
  honest[rep] <- mean(sapply(1:5, function(j) { tr <- f != j; s <- select_k(Xn[tr, ], yn[tr], 20)   # selección dentro de la partición
    mean(knn(Xn[tr, s], Xn[!tr, s], factor(yn[tr]), 5) == yn[!tr]) }))
}
cat(sprintf("feature selection on NOISE (true accuracy 0.5): selected before CV %.3f | selected inside the loop %.3f\n", mean(leaky), mean(honest)))
cv5 <- strat_kfold(y_pos, 5, 0)
leaky_sc <- mean(sapply(cv5, function(f) mean(knn(scale(X)[f$train, ], scale(X)[f$test, ], factor(y_pos[f$train]), 5) == y_pos[f$test])))   # escalado con todos los datos
honest_sc <- mean(sapply(cv5, function(f) { sc <- scale(X[f$train, ]); te <- scale(X[f$test, ], attr(sc, "scaled:center"), attr(sc, "scaled:scale"))
  mean(knn(sc, te, factor(y_pos[f$train]), 5) == y_pos[f$test]) }))
cat(sprintf("scaling on the breast-cancer data: scaled before CV %.4f | scaled inside the fold %.4f\n", leaky_sc, honest_sc))



En datos sin señal, seleccionar las 20 "mejores" de 1,000 características de ruido antes de la validación cruzada da una exactitud muy por encima del azar —un resultado que pasaría una revisión por pares—, mientras que la misma selección dentro del ciclo da el valor honesto, cercano a 0.5. La fuga por escalamiento casi no cambia la exactitud en cáncer de mama: el escalador aprende dos números por característica, que la partición de prueba apenas mueve. El daño es proporcional a cuánto aprende el preprocesamiento de las etiquetas; reajustar dentro del ciclo no cuesta nada y cubre ambos casos.

### 7.2 Características derivadas de la variable objetivo

La segunda fuga es una característica que es consecuencia del desenlace y no un predictor de él: el tratamiento prescrito después del diagnóstico, un examen solicitado porque el clínico ya sospechaba la enfermedad, un código de facturación generado por el propio evento que se predice. Estas características son extremadamente predictivas en la base de datos retrospectiva y no están disponibles en el momento de la predicción. La simulación añade un "código de tratamiento posbiopsia" que coincide con el resultado de la biopsia en el 95 % de los casos.



In [ ]:
# Fuga 2: una característica derivada del desenlace (disponible en la base de datos, no en el momento de la predicción).
set.seed(1); treatment_code <- ifelse(runif(nrow(X)) < 0.95, y_pos, 1 - y_pos)   # el 'tratamiento administrado' coincide con la biopsia el 95% de las veces
Xleak <- cbind(Xs, treatment = treatment_code)
cv_acc <- function(Xm) mean(sapply(strat_kfold(y_pos, 10, 0), function(f) {
  fit <- suppressWarnings(glm(y ~ ., data.frame(Xm, y = y_pos)[f$train, ], family = binomial))
  mean((predict(fit, data.frame(Xm, y = y_pos)[f$test, ], type = "response") >= 0.5) == y_pos[f$test]) }))
honest_acc <- cv_acc(Xs); leaky_acc <- cv_acc(Xleak)
coefs <- coef(suppressWarnings(glm(y ~ ., data.frame(Xleak, y = y_pos), family = binomial)))[-1]
rank <- which(order(-abs(coefs)) == length(coefs))
cat(sprintf("CV accuracy without the leaked feature: %.3f   with it: %.3f\n", honest_acc, leaky_acc))
cat(sprintf("the leaked feature has the largest-ranked standardised coefficient #%d of %d (|coef| = %.2f, next largest %.2f)\n",
            rank, length(coefs), abs(coefs[length(coefs)]), sort(abs(coefs), decreasing = TRUE)[2]))



La exactitud sube y el modelo se apoya en la columna filtrada más que en cualquier medición nuclear. La validación cruzada no puede detectarlo, porque la fuga está presente en todas las particiones. La defensa es clínica, no estadística —para cada característica, pregúntate <em>¿se conoce este valor en el momento en que se necesita la predicción?</em>—, y el ítem de TRIPOD+AI sobre el momento de medición de cada predictor existe por esta razón (Sección 8).

### 7.3 Mediciones repetidas por paciente

La tercera fuga es estructural. Cuando los pacientes aportan varios registros —visitas, imágenes, segmentos de ECG—, la división aleatoria pone algunos registros de un paciente en entrenamiento y otros en prueba. El modelo entonces reconoce al <em>paciente</em>, no la condición. Saeb y colaboradores mostraron que la validación cruzada por registro de datos de sensores clínicos infla drásticamente la exactitud respecto de la validación cruzada por sujeto, y que la mayoría de los estudios publicados había usado la equivocada.<sup class="c"><a href="#r25">[25]</a></sup> El remedio es una división agrupada (Figura 3). La simulación construye 100 pacientes con 8 registros cada uno; cada paciente tiene una línea base individual en todas las características (lo que hace reconocibles sus registros) y la enfermedad desplaza moderadamente tres características.



In [ ]:
# Fuga 3: varios registros por paciente, divididos al azar (por registro) frente a por paciente (agrupados).
set.seed(0); n_pat <- 100; n_rec <- 8
label_pat <- sample(0:1, n_pat, replace = TRUE)                       # un diagnóstico por paciente
baseline_pat <- matrix(rnorm(n_pat * 10, 0, 1.5), n_pat, 10)          # línea base propia de cada paciente en 10 características
effect <- 1.5 * c(rep(1, 3), rep(0, 7))                               # la enfermedad desplaza 3 de las 10 características
X_rep <- do.call(rbind, lapply(1:n_pat, function(p) t(sapply(1:n_rec, function(i) baseline_pat[p, ] + effect * label_pat[p] + rnorm(10)))))
y_rep <- rep(label_pat, each = n_rec); patient <- rep(1:n_pat, each = n_rec)
sc_cv <- function(folds) mean(sapply(folds, function(f) { sc <- scale(X_rep[f$train, ]); te <- scale(X_rep[f$test, ], attr(sc, "scaled:center"), attr(sc, "scaled:scale"))
  mean(knn(sc, te, factor(y_rep[f$train]), 5) == y_rep[f$test]) }))
record_folds <- strat_kfold(y_rep, 5, 0)
set.seed(0); pat_fold <- sample(rep(1:5, length.out = n_pat))[patient]
group_folds <- lapply(1:5, function(j) list(train = which(pat_fold != j), test = which(pat_fold == j)))
cat(sprintf("%d records from %d patients\n", length(y_rep), n_pat))
cat(sprintf("record-wise 5-fold CV (patients leak across folds): %.3f\n", sc_cv(record_folds)))
cat(sprintf("patient-wise 5-fold CV (grouped):                   %.3f\n", sc_cv(group_folds)))



La validación cruzada por registro informa una exactitud muy superior a la de la validación por paciente. La primera responde "¿puede el modelo clasificar un registro nuevo de un paciente que ya ha visto?" —casi nunca la pregunta clínica. La segunda responde "¿puede clasificar a un paciente nuevo?", y es mucho menor porque el efecto de la enfermedad es moderado en relación con la variación entre pacientes. La misma lógica se aplica a cualquier agrupación que no se repetirá en el despliegue: dividir por hospital para estimar el desempeño en un hospital nuevo, por año para estimar el desempeño del año siguiente.

<div class="box warn"><h4>La fuga de datos es silenciosa</h4><p>Ninguna de las tres fugas produce una advertencia ni un número inverosímil: las exactitudes con fuga de arriba son valores que un revisor aceptaría. Se descubren razonando sobre el proceso que generó los datos, no inspeccionando la salida. La lista de verificación de Kapoor y Narayanan —¿separación limpia entre entrenamiento y prueba en cada paso de preprocesamiento? ¿alguna característica no disponible en el momento de la predicción? ¿duplicados o grupos a ambos lados de la división? ¿la distribución de prueba coincide con la del despliegue?— debería aplicarse a cada evaluación antes de creer su número.<sup class="c"><a href="#r12">[12]</a></sup></p></div>

## 8. Validación interna frente a externa, e informe de resultados

Todo lo anterior es <strong>validación interna</strong>: los casos de prueba, por cuidadosamente que se hayan reservado, provienen de la misma fuente que los casos de entrenamiento —el mismo hospital, escáneres, años, criterios de inclusión, etiquetado. Estima el desempeño <em>en esa fuente</em>. El desempeño en otro lugar es una cantidad distinta, que solo se estima mediante <strong>validación externa</strong>: aplicar el modelo congelado, sin reajustarlo, a datos recolectados en otro lugar, más tarde o con otros criterios.<sup class="c"><a href="#r14">[14]</a>,<a href="#r28">[28]</a></sup> Revela lo que la validación interna no puede: el shortcut learning (Unidad 1, donde un identificador del escáner predecía la etiqueta en un hospital y no en otros), descalibración por una prevalencia o una mezcla de casos distinta, y deterioro por diferencias entre poblaciones. El modelo de sepsis de la Unidad 1 es el caso canónico: números internos excelentes, una sensibilidad de 33 % en la validación externa.<sup class="c"><a href="#r32">[32]</a></sup>

El bloque de código simula esta estructura con los datos de cáncer de mama, usando una característica como sustituto del "centro": el modelo se desarrolla con los tumores cuyo <em>recuento mitótico</em> está bajo la mediana (un "hospital", donde predominan los tumores en etapa temprana) y se aplica, congelado, a los que están sobre la mediana (otro "hospital", que ve sobre todo tumores avanzados, con más frecuencia malignos).



In [ ]:
# Validación interna (validación cruzada, mismo 'hospital') frente a externa (modelo congelado, otro 'hospital').
site <- X[, "Mitoses"] > median(X[, "Mitoses"])                     # FALSE = centro de desarrollo, TRUE = centro externo
dev <- !site; ext <- site
prob_int <- cvprob_glm(as.matrix(Xs[dev, ]), y_pos[dev])            # interna: CV de 10 particiones en el centro de desarrollo
fit_dev <- suppressWarnings(glm(y ~ ., data.frame(Xs[dev, ], y = y_pos[dev]), family = binomial))
prob_ext <- predict(fit_dev, data.frame(Xs[ext, ]), type = "response")   # modelo congelado, nunca reajustado
report <- function(yt, pr) { pred <- as.integer(pr >= 0.5)
  tp <- sum(pred == 1 & yt == 1); fn <- sum(pred == 0 & yt == 1); fp <- sum(pred == 1 & yt == 0); tn <- sum(pred == 0 & yt == 0)
  round(c(prevalence = mean(yt), accuracy = (tp + tn) / length(yt), sensitivity = tp / (tp + fn), specificity = tn / (tn + fp),
          AUC = as.numeric(auc(yt, pr, quiet = TRUE)), Brier = mean((pr - yt)^2), `obs/exp` = mean(yt) / mean(pr)), 3) }
cat(sprintf("development site: %d tumours | external site: %d tumours\n", sum(dev), sum(ext)))
as.data.frame(rbind(`internal (CV, development site)` = report(y_pos[dev], prob_int), `external (frozen model, other site)` = report(y_pos[ext], prob_ext)))



El patrón es el que advierte Van Calster.<sup class="c"><a href="#r10">[10]</a></sup> La discriminación sobrevive: el AUC externo sigue siendo alto (aproximadamente 0.95 frente a 0.99 interno), de modo que el modelo sigue ordenando los malignos por encima de los benignos, y la calibration-in-the-large también se mantiene: la razón observado/esperado sigue cerca de 1. Lo que no sobrevive es el punto de operación del umbral congelado. En el centro externo, donde los tumores tienen recuentos mitóticos altos y la malignidad es mucho más frecuente, el mismo punto de corte de 0.5 envía a biopsia muchos más tumores benignos: la especificidad cae de aproximadamente 0.98 a 0.85 mientras la sensibilidad sube, y el puntaje de Brier empeora respecto de la estimación interna; la exactitud apenas se mueve, solo porque el conjunto externo es ahora aproximadamente 89 % maligno, de modo que ha dejado de ser un resumen informativo. Un informe que entregara solo el AUC calificaría a este modelo de transportable; uno que entregara las métricas dependientes del umbral a la prevalencia del despliegue exigiría volver a derivar el umbral antes del despliegue —la conclusión correcta. Lo general es que un umbral congelado no sobrevive a un cambio en la mezcla de casos, aunque la discriminación y la calibration-in-the-large sí lo hagan.

### 8.1 Qué informar: TRIPOD+AI

La declaración <strong>TRIPOD+AI</strong> (Transparent Reporting of a multivariable prediction model for Individual Prognosis Or Diagnosis, actualización de 2024 para aprendizaje automático) es el estándar de informe que las revistas médicas exigen para los modelos de predicción.<sup class="c"><a href="#r13">[13]</a></sup> En el vocabulario de este suplemento, sus ítems relativos a la evaluación son:

- **Fuente de los datos y contexto** del desarrollo y de cada validación; fechas; criterios de elegibilidad —para que el lector pueda juzgar <em>de qué</em> es muestra la muestra de prueba.
- Definición y momento del **desenlace**; **predictores**, con el momento en que se mide cada uno respecto del desenlace (la defensa frente a la Sección 7.2).
- **Tamaño muestral** y su justificación; Riley y colaboradores dan el cálculo,<sup class="c"><a href="#r27">[27]</a></sup> y la fórmula binomial de la Sección 2.1 es la versión rápida, de servilleta.
- **Desarrollo del modelo**: todos los modelos candidatos e hiperparámetros probados, y cómo se seleccionaron (Sección 3.4). **Datos faltantes** tratados dentro del ciclo.
- **Método de validación**: interna (qué remuestreo, cuántas particiones, anidada o no, agrupada según qué) y externa (qué población).
- **Medidas de desempeño**: discriminación (AUC con un intervalo de confianza), calibración (gráfico y resumen), medidas dependientes del umbral en un umbral <em>preespecificado</em> junto con la prevalencia de la muestra (Secciones 5.3–5.6); en regresión, el error en las unidades de la variable objetivo (Sección 6). El marco de Steyerberg las organiza.<sup class="c"><a href="#r26">[26]</a></sup>
- **Equidad**: desempeño en subgrupos relevantes (sexo, edad, etnia, centro), el añadido de TRIPOD+AI que conecta con la Unidad 1. **Disponibilidad** del código, los datos y el modelo.

Relee el ejemplo de shortcut learning de la Unidad 1 con esta lista. El modelo con la característica del escáner informó una exactitud de entrenamiento (Sección 1.2: no es una medición), una exactitud en casos nuevos del mismo hospital (validación interna) y una exactitud en un hospital nuevo (validación externa), y solo la tercera reveló la falla. Un informe conforme a TRIPOD+AI también habría enumerado el "identificador del escáner" entre los predictores, con su momento de medición, y un lector se habría preguntado por qué un escáner debería predecir un diagnóstico.

<div class="box bridge"><h4>Puente hacia los talleres del curso</h4><p>El Taller 2 te pide presentar un artículo que aplique aprendizaje supervisado. Usa la lista anterior como pauta de lectura: para cada ítem, anota qué informa el artículo y qué omite. Un artículo que informa la exactitud sin el tamaño de prueba, una métrica dependiente del umbral sin la prevalencia, o un AUC sin calibración no ha aportado la evidencia que necesita un clínico; decirlo con precisión es la habilidad que evalúa el taller.</p></div>

## 9. Una función de evaluación reutilizable

Las unidades posteriores necesitan, una y otra vez, la misma evaluación: validación cruzada estratificada repetida de varios clasificadores con las métricas de la Sección 5, el AUC y el puntaje de Brier, y una medida de incertidumbre. La función siguiente la empaqueta; cópiala en cualquier notebook. Recibe una lista de wrappers de modelos (cada uno devuelve probabilidades out-of-fold de la clase positiva), datos con la clase positiva codificada como 1 y una asignación de particiones (por defecto, 5 particiones estratificadas repetidas 5 veces), y devuelve la media y la SD entre particiones para cada métrica. Todos los modelos se comparan en las <em>mismas</em> particiones, lo que elimina de la comparación el ruido entre divisiones de la Figura 2.



In [ ]:
# Una función de evaluación reutilizable para clasificadores binarios (copia este bloque de código en notebooks posteriores).
evaluate_classifiers <- function(models, Xm, y, k = 5, repeats = 5, threshold = 0.5) {
  # models: lista nombrada de funciones (Xtr, ytr, Xte) -> P(positivo) en Xte;  y: 0/1 con 1 = condición que se busca detectar.
  recs <- list()
  for (rp in 1:repeats) { folds <- strat_kfold(y, k, seed = rp)
    for (nm in names(models)) for (f in folds) {
      prob <- models[[nm]](Xm[f$train, , drop = FALSE], y[f$train], Xm[f$test, , drop = FALSE])
      pred <- as.integer(prob >= threshold); yt <- y[f$test]
      tp <- sum(pred == 1 & yt == 1); fn <- sum(pred == 0 & yt == 1); fp <- sum(pred == 1 & yt == 0); tn <- sum(pred == 0 & yt == 0)
      m <- metrics_from_counts(tp, fn, fp, tn)
      recs[[length(recs) + 1]] <- data.frame(model = nm, accuracy = unname(m["accuracy"]), sensitivity = unname(m["sensitivity (recall)"]),
        specificity = unname(m["specificity"]), AUC = as.numeric(auc(yt, prob, quiet = TRUE)), Brier = mean((prob - yt)^2)) }
  }
  long <- do.call(rbind, recs)
  agg <- aggregate(cbind(accuracy, sensitivity, specificity, AUC, Brier) ~ model, long, function(z) c(mean = mean(z), sd = sd(z)))
  do.call(data.frame, agg)
}
m_glm <- function(Xtr, ytr, Xte) { fit <- suppressWarnings(glm(y ~ ., data.frame(Xtr, y = ytr), family = binomial)); predict(fit, data.frame(Xte), type = "response") }
m_knn <- function(Xtr, ytr, Xte) { pr <- knn(Xtr, Xte, factor(ytr), k = 5, prob = TRUE); w <- attr(pr, "prob"); ifelse(pr == 1, w, 1 - w) }
m_nb  <- function(Xtr, ytr, Xte) predict(naiveBayes(yf ~ ., data.frame(Xtr, yf = factor(ytr))), data.frame(Xte), type = "raw")[, 2]
results <- evaluate_classifiers(list("logistic regression" = m_glm, "5-nearest neighbours" = m_knn, "Gaussian naive Bayes" = m_nb), Xs, y_pos)
round_df <- results; num <- sapply(round_df, is.numeric); round_df[num] <- lapply(round_df[num], round, 3); round_df



La tabla se lee como un resumen al estilo TRIPOD: discriminación (AUC), calibración (Brier) y las dos tasas clínicas en el umbral preespecificado, cada una con su SD entre particiones, sobre particiones idénticas. Las diferencias menores que aproximadamente una SD entre particiones no deben sobreinterpretarse; la diferencia de Brier entre Bayes ingenuo (aproximadamente 0.037) y los demás (aproximadamente 0.025) es mayor que esa dispersión y, promediada sobre las 25 particiones, es una diferencia real de calibración. Las Unidades 3, 4 y 6 llaman a una función de esta forma con sus propias listas de modelos.

<div class="box"><h4>Ejercicio 5</h4><p>Llama a la función con las particiones agrupadas y los datos de mediciones repetidas de la Sección 7.3 (reemplaza <code>strat_kfold</code> en su interior por la asignación agrupada), y con <code>threshold = 0.09</code> sobre los datos de cáncer de mama (el umbral derivado de los costos de la Sección 5.5). Explica cada columna que cambia entre el umbral 0.5 y el 0.09, y la única columna que no cambia.</p></div>

## 10. Resumen

Ahora deberías poder explicar, sin apuntes:

- Por qué el error de entrenamiento está sesgado de forma optimista, y los roles de las muestras de entrenamiento, validación y prueba (y que la muestra de prueba se usa una sola vez).
- Por qué una única exactitud hold-out es un sorteo de una distribución cuyo ancho está determinado por el tamaño de prueba, y de qué protege la estratificación.
- Cómo funciona, a mano, la validación cruzada k-fold; cuándo son apropiadas las variantes estratificada, repetida, dejar uno fuera y agrupada; y por qué seleccionar hiperparámetros e informar el desempeño en las mismas particiones es optimista (CV anidada).
- El sobreajuste y el subajuste en términos de la complejidad del modelo; la descomposición sesgo–varianza, su total en forma de U, y cómo lo muestran la curva de complejidad y la curva de aprendizaje; la maldición de la dimensionalidad y la razón \(p/n\).
- La matriz de confusión y cada métrica derivada de ella, con sus nombres de aprendizaje automático y clínicos; por qué la exactitud falla ante el desbalance; por qué el PPV depende de la prevalencia (Bayes) mientras que la sensibilidad y la especificidad no.
- Cómo se construye una curva ROC a partir de puntajes, qué significa el AUC como probabilidad, cuándo es más informativa la curva de precisión–sensibilidad, cómo se deduce un umbral de los costos de los dos errores, y qué es la calibración y cómo evaluarla.
- MSE, RMSE, MAE y \(R^2\), y por qué \(R^2\) es negativo cuando un modelo es peor que la media.
- Las tres fugas clásicas —preprocesar antes de dividir, características derivadas de la variable objetivo, pacientes repetidos entre particiones—, cómo cada una infla un número y cómo se previene cada una.
- La diferencia entre validación interna y externa, qué estima cada una, y los ítems de TRIPOD+AI que debe contener el informe de un modelo.

## 11. Autoevaluación

<details><summary>1. Un colega informa "nuestra red neuronal alcanza 98 % de exactitud en el conjunto de entrenamiento". Enumera las tres preguntas que harías antes de tomar en serio el número, y di qué enseña un clasificador 1-NN sobre la primera de ellas.</summary>¿En qué casos se midió (entrenamiento, validación o prueba)? ¿Cómo se separaron de los datos de entrenamiento (división aleatoria, por paciente, por hospital, por fecha)? ¿Cuántos eran? En el conjunto de entrenamiento, 1-NN alcanza esencialmente 100 % por construcción, sean cuales sean los datos, de modo que la exactitud de entrenamiento no es una medición de nada.</details>

<details><summary>2. Un conjunto de prueba tiene 50 casos y la exactitud medida es 0.90. Da el error estándar aproximado y di si se ha demostrado que un segundo modelo con 0.94 en el mismo conjunto de prueba es mejor.</summary>\(\sqrt{0.9 \times 0.1 / 50} \approx 0.042\). Una diferencia de 0.04 equivale a aproximadamente un error estándar; no se ha demostrado. Se necesita una comparación pareada en las mismas particiones, o un conjunto de prueba mucho más grande.</details>

<details><summary>3. Escribe los índices de prueba de las cuatro particiones de un k-fold sin barajar sobre 10 casos (índices 1–10), con bloques de tamaño casi igual.</summary>{1, 2, 3}, {4, 5, 6}, {7, 8}, {9, 10} (o una partición similar de tamaños casi iguales). El conjunto de entrenamiento de cada partición es el complemento. Si los casos estuvieran ordenados por clase, la primera partición se entrenaría con una muestra empobrecida en su propia clase, y por eso se usa un barajado.</details>

<details><summary>4. Explica en una oración cada cosa: por qué dejar uno fuera tiene bajo sesgo y alta varianza, y cuándo se prefiere k-fold estratificado sobre k-fold simple.</summary>Bajo sesgo porque cada modelo se entrena con \(n-1\) casos, casi la muestra completa; alta varianza porque los \(n\) conjuntos de entrenamiento se superponen casi por completo, de modo que los \(n\) errores están muy correlacionados y su promedio fluctúa con la muestra particular. Se prefiere k-fold estratificado cuando el conjunto de datos es pequeño o desbalanceado, para que cada partición contenga la misma proporción de cada clase.</details>

<details><summary>5. En el experimento de CV anidada sobre ruido, el mejor puntaje interno estuvo por encima de 0.5 y la estimación anidada fue de aproximadamente 0.5. ¿Qué número aparecería en un artículo que ajustó en el conjunto de prueba, y qué principio general ilustra la diferencia?</summary>El mejor puntaje interno: el máximo de muchas estimaciones ruidosas, sesgado hacia arriba. El principio: los datos usados para tomar una decisión (modelo, hiperparámetro, característica, umbral) no pueden usarse también para informar el desempeño de lo elegido; por cada nivel de mirar se requiere un nivel más de reserva.</details>

<details><summary>6. Define sesgo y varianza, di cuál conduce al subajuste y cuál al sobreajuste, y nombra la perilla que mueve a k-NN entre ambos.</summary>El sesgo es el error que proviene de supuestos erróneos del modelo (una forma demasiado simple) y conduce al subajuste; la varianza es la sensibilidad a pequeñas fluctuaciones del conjunto de entrenamiento y conduce al sobreajuste. La perilla es \(k\): un \(k\) pequeño da bajo sesgo y alta varianza; un \(k\) grande, lo contrario.</details>

<details><summary>7. Calcula a mano, para TP = 40, FN = 10, FP = 20, TN = 930: exactitud, sensibilidad, especificidad, PPV, NPV, F1 y exactitud balanceada.</summary>Exactitud 970/1000 = 0.970; sensibilidad 40/50 = 0.80; especificidad 930/950 ≈ 0.979; PPV 40/60 ≈ 0.667; NPV 930/940 ≈ 0.989; F1 = 80/(80 + 20 + 10) ≈ 0.727; exactitud balanceada ≈ 0.889. La exactitud está dominada por los 950 negativos.</details>

<details><summary>8. Una prueba tiene sensibilidad 0.95 y especificidad 0.95. Calcula su PPV con 50 % de prevalencia y con 1 % de prevalencia, y explica en una oración por qué difieren.</summary>Con 50 %: \(0.95 \times 0.5 / (0.95 \times 0.5 + 0.05 \times 0.5) = 0.95\). Con 1 %: \(0.0095 / (0.0095 + 0.0495) \approx 0.161\). Con baja prevalencia, las falsas alarmas, que provienen del gran grupo de sanos, superan en número a los verdaderos positivos, que provienen del pequeño grupo de enfermos.</details>

<details><summary>9. Los puntajes de 4 pacientes enfermos son 0.8, 0.6, 0.5, 0.2 y los de 3 pacientes sanos 0.7, 0.4, 0.1. Calcula el AUC contando pares.</summary>12 pares. 0.8 supera a 3; 0.6 supera a 2 (0.4, 0.1); 0.5 supera a 2; 0.2 supera a 1 (0.1). AUC = 8/12 ≈ 0.667.</details>

<details><summary>10. Dos modelos tienen el mismo AUC de 0.90. Uno tiene un puntaje de Brier de 0.08 y el otro de 0.20. ¿Qué concluyes, y cuál de los dos puede usarse con la regla del umbral basada en costos \(t^{*} = c_{FP}/(c_{FP} + c_{FN})\)?</summary>Ordenan a los pacientes igual de bien, pero el segundo está mal calibrado: sus probabilidades no coinciden con las frecuencias observadas. La regla del umbral supone probabilidades calibradas, de modo que solo el primero puede usarla directamente; el segundo debe recalibrarse primero (escalado de Platt, regresión isotónica).</details>

<details><summary>11. Lee este código y di qué tiene de incorrecto y qué número inflará:<br><code>sel <- select_k(X, y, 10)</code><br><code>acc <- cv_svm(X[, sel], y, folds = 5)</code></summary>La selección de características se ajustó con todos los datos, incluidos los casos que cada partición usa después para la prueba, de modo que las características seleccionadas ya "vieron" las etiquetas de prueba. La exactitud de CV queda inflada: en ruido puro sube muy por encima de 0.5. La selección debe repetirse dentro de cada partición de entrenamiento.</details>

<details><summary>12. Un modelo desarrollado en un hospital terciario tiene un AUC externo igual a su AUC interno, pero su sensibilidad en el umbral preespecificado baja de 0.95 a 0.80 y su razón observado/esperado es 1.8. ¿Qué ocurrió y qué debe hacerse antes del despliegue?</summary>La discriminación se conserva, pero la calibración no: la población externa tiene una prevalencia o una mezcla de casos distinta, de modo que las probabilidades del modelo están descalibradas y menos casos superan el umbral. Antes del despliegue, el modelo debe recalibrarse con datos locales (al menos el intercepto, la calibration-in-the-large) y el umbral debe volver a deducirse de los costos locales, y el informe debe presentar ambas validaciones.</details>

## Glosario

<dl class="gloss">
<dt>Área bajo la curva ROC (area under the ROC curve, AUC)</dt><dd>Probabilidad de que un caso positivo elegido al azar reciba un puntaje más alto que un caso negativo elegido al azar; 1 = ordenamiento perfecto, 0.5 = azar.</dd>
<dt>Exactitud balanceada (balanced accuracy)</dt><dd>Media de la sensibilidad y la especificidad; la exactitud que tendría el clasificador si ambas clases fueran igual de frecuentes.</dd>
<dt>Sesgo / varianza (bias / variance)</dt><dd>Error sistemático del ajuste promedio debido a supuestos erróneos del modelo (conduce al subajuste) / sensibilidad del ajuste a las fluctuaciones de la muestra de entrenamiento (conduce al sobreajuste). Error cuadrático esperado = sesgo² + varianza + ruido.</dd>
<dt>Puntaje de Brier (Brier score)</dt><dd>Diferencia cuadrática media entre la probabilidad predicha y el desenlace observado; mide conjuntamente la calibración y la discriminación.</dd>
<dt>Calibración (calibration)</dt><dd>Concordancia entre las probabilidades predichas y las frecuencias observadas; se evalúa con un diagrama de confiabilidad y la razón observado/esperado.</dd>
<dt>Matriz de confusión (confusion matrix)</dt><dd>Tabla cruzada de la clase verdadera frente a la predicha: TP, FN, FP, TN.</dd>
<dt>Validación cruzada k-fold (k-fold cross-validation)</dt><dd>División de la muestra en k particiones; cada partición se evalúa una vez con un modelo entrenado en las otras k−1; los k puntajes se promedian. Variantes: estratificada, repetida, dejar uno fuera (k = n), agrupada.</dd>
<dt>Maldición de la dimensionalidad (curse of dimensionality)</dt><dd>La dispersión de una muestra fija en un espacio de características cuya dimensión crece; las distancias pierden sentido y la razón parámetros/observaciones p/n aumenta.</dd>
<dt>Fuga de datos (data leakage)</dt><dd>Uso durante el entrenamiento de información no disponible en el momento de la predicción (estadísticos del conjunto de prueba, características derivadas del desenlace, registros de pacientes de prueba), que infla la evaluación.</dd>
<dt>Validación externa (external validation)</dt><dd>Evaluación de un modelo congelado con datos de una fuente (centro, período, población) distinta de la de los datos de desarrollo.</dd>
<dt>Puntaje F1 (F1 score)</dt><dd>Media armónica de la precisión y la sensibilidad con igual peso; ignora los verdaderos negativos.</dd>
<dt>Error de generalización (generalisation error)</dt><dd>Pérdida esperada de un modelo en casos nuevos de la población; la cantidad que estima todo método de evaluación.</dd>
<dt>Hold-out</dt><dd>Una única división aleatoria en una parte de entrenamiento y una de prueba; su estimación tiene una varianza determinada por el tamaño de prueba.</dd>
<dt>Hiperparámetro (hyperparameter)</dt><dd>Un parámetro de configuración del algoritmo de aprendizaje que no se aprende de los datos de entrenamiento (k en k-NN, profundidad del árbol, intensidad de la regularización), elegido con datos de validación o validación cruzada.</dd>
<dt>Desbalance (imbalance)</dt><dd>Frecuencias de clase desiguales; leve (minoritaria 20–40 %), moderado (1–20 %), extremo (< 1 %). La exactitud deja de ser informativa.</dd>
<dt>Coeficiente de correlación de Matthews (Matthews correlation coefficient, MCC)</dt><dd>Correlación entre las clases verdaderas y las predichas calculada a partir de las cuatro celdas de la matriz de confusión; de −1 a +1, 0 al azar.</dd>
<dt>Validación cruzada anidada (nested cross-validation)</dt><dd>Un ciclo externo de CV para la estimación en torno a un ciclo interno de CV para la selección de hiperparámetros, de modo que ninguna partición de prueba se usa para ninguna decisión.</dd>
<dt>Sobreajuste / subajuste (overfitting / underfitting)</dt><dd>Ajustar los accidentes de la muestra de entrenamiento por exceso de complejidad (puntaje de entrenamiento alto, puntaje de prueba bajo) / no lograr ajustar la regularidad por complejidad insuficiente (ambos bajos).</dd>
<dt>Precisión (precision, PPV) / NPV</dt><dd>Fracción de los predichos positivos que son realmente positivos / de los predichos negativos que son realmente negativos; ambas dependen de la prevalencia.</dd>
<dt>Curva de precisión–sensibilidad (precision–recall curve)</dt><dd>Precisión frente a sensibilidad sobre todos los umbrales; su línea base es igual a la prevalencia; preferida en problemas desbalanceados. Se resume con la precisión promedio.</dd>
<dt>Coeficiente de determinación (R²)</dt><dd>Uno menos la razón entre el error cuadrático del modelo y el de predecir la media; negativo en un conjunto de prueba cuando el modelo es peor que la media.</dd>
<dt>Curva ROC (ROC curve)</dt><dd>Tasa de verdaderos positivos frente a tasa de falsos positivos sobre todos los umbrales; invariante a la prevalencia.</dd>
<dt>Sensibilidad (sensitivity, recall, TPR) / especificidad (specificity, TNR)</dt><dd>Fracción de los verdaderos positivos detectados / de los verdaderos negativos descartados; propiedades de la prueba, independientes de la prevalencia.</dd>
<dt>Estratificación (stratification)</dt><dd>Dividir dentro de cada clase para que cada parte tenga las mismas proporciones de clases.</dd>
<dt>Umbral (threshold)</dt><dd>El puntaje por encima del cual un caso se declara positivo; lo fijan los costos relativos de los falsos negativos y los falsos positivos, \(t^{*} = c_{FP}/(c_{FP}+c_{FN})\) para probabilidades calibradas.</dd>
<dt>TRIPOD+AI</dt><dd>La declaración de 2024 sobre el informe de modelos de predicción en medicina, incluidos los modelos de aprendizaje automático.</dd>
</dl>

## Referencias y lecturas recomendadas

<ol class="refs">
<li id="r1">Stone M. Cross-validatory choice and assessment of statistical predictions. <em>J R Stat Soc Ser B</em>. 1974;36(2):111–147. <a href="https://doi.org/10.1111/j.2517-6161.1974.tb00994.x">doi:10.1111/j.2517-6161.1974.tb00994.x</a></li>
<li id="r2">Kohavi R. A study of cross-validation and bootstrap for accuracy estimation and model selection. In: <em>Proceedings of the 14th International Joint Conference on Artificial Intelligence (IJCAI-95)</em>. Morgan Kaufmann; 1995:1137–1145. Sin DOI; actas disponibles en <a href="https://www.ijcai.org/Proceedings/95-2/Papers/016.pdf">ijcai.org/Proceedings/95-2/Papers/016.pdf</a></li>
<li id="r3">Varma S, Simon R. Bias in error estimation when using cross-validation for model selection. <em>BMC Bioinformatics</em>. 2006;7:91. <a href="https://doi.org/10.1186/1471-2105-7-91">doi:10.1186/1471-2105-7-91</a></li>
<li id="r4">Hanley JA, McNeil BJ. The meaning and use of the area under a receiver operating characteristic (ROC) curve. <em>Radiology</em>. 1982;143(1):29–36. <a href="https://doi.org/10.1148/radiology.143.1.7063747">doi:10.1148/radiology.143.1.7063747</a></li>
<li id="r5">Fawcett T. An introduction to ROC analysis. <em>Pattern Recognit Lett</em>. 2006;27(8):861–874. <a href="https://doi.org/10.1016/j.patrec.2005.10.010">doi:10.1016/j.patrec.2005.10.010</a></li>
<li id="r6">Saito T, Rehmsmeier M. The precision-recall plot is more informative than the ROC plot when evaluating binary classifiers on imbalanced datasets. <em>PLoS ONE</em>. 2015;10(3):e0118432. <a href="https://doi.org/10.1371/journal.pone.0118432">doi:10.1371/journal.pone.0118432</a></li>
<li id="r7">Matthews BW. Comparison of the predicted and observed secondary structure of T4 phage lysozyme. <em>Biochim Biophys Acta</em>. 1975;405(2):442–451. <a href="https://doi.org/10.1016/0005-2795(75)90109-9">doi:10.1016/0005-2795(75)90109-9</a></li>
<li id="r8">Chicco D, Jurman G. The advantages of the Matthews correlation coefficient (MCC) over F1 score and accuracy in binary classification evaluation. <em>BMC Genomics</em>. 2020;21:6. <a href="https://doi.org/10.1186/s12864-019-6413-7">doi:10.1186/s12864-019-6413-7</a></li>
<li id="r9">Brier GW. Verification of forecasts expressed in terms of probability. <em>Mon Weather Rev</em>. 1950;78(1):1–3. <a href="https://doi.org/10.1175/1520-0493(1950)078<0001:VOFEIT>2.0.CO;2">doi:10.1175/1520-0493(1950)078&lt;0001:VOFEIT&gt;2.0.CO;2</a></li>
<li id="r10">Van Calster B, McLernon DJ, van Smeden M, Wynants L, Steyerberg EW. Calibration: the Achilles heel of predictive analytics. <em>BMC Med</em>. 2019;17:230. <a href="https://doi.org/10.1186/s12916-019-1466-7">doi:10.1186/s12916-019-1466-7</a></li>
<li id="r11">Kaufman S, Rosset S, Perlich C, Stitelman O. Leakage in data mining: formulation, detection, and avoidance. <em>ACM Trans Knowl Discov Data</em>. 2012;6(4):15. <a href="https://doi.org/10.1145/2382577.2382579">doi:10.1145/2382577.2382579</a></li>
<li id="r12">Kapoor S, Narayanan A. Leakage and the reproducibility crisis in machine-learning-based science. <em>Patterns</em>. 2023;4(9):100804. <a href="https://doi.org/10.1016/j.patter.2023.100804">doi:10.1016/j.patter.2023.100804</a></li>
<li id="r13">Collins GS, Moons KGM, Dhiman P, et al. TRIPOD+AI statement: updated guidance for reporting clinical prediction models that use regression or machine learning methods. <em>BMJ</em>. 2024;385:e078378. <a href="https://doi.org/10.1136/bmj-2023-078378">doi:10.1136/bmj-2023-078378</a></li>
<li id="r14">Steyerberg EW, Vergouwe Y. Towards better clinical prediction models: seven steps for development and an ABCD for validation. <em>Eur Heart J</em>. 2014;35(29):1925–1931. <a href="https://doi.org/10.1093/eurheartj/ehu207">doi:10.1093/eurheartj/ehu207</a></li>
<li id="r15">Geman S, Bienenstock E, Doursat R. Neural networks and the bias/variance dilemma. <em>Neural Comput</em>. 1992;4(1):1–58. <a href="https://doi.org/10.1162/neco.1992.4.1.1">doi:10.1162/neco.1992.4.1.1</a></li>
<li id="r16">Altman DG, Bland JM. Diagnostic tests 1: sensitivity and specificity. <em>BMJ</em>. 1994;308(6943):1552. <a href="https://doi.org/10.1136/bmj.308.6943.1552">doi:10.1136/bmj.308.6943.1552</a></li>
<li id="r17">Altman DG, Bland JM. Diagnostic tests 2: predictive values. <em>BMJ</em>. 1994;309(6947):102. <a href="https://doi.org/10.1136/bmj.309.6947.102">doi:10.1136/bmj.309.6947.102</a></li>
<li id="r18">Hastie T, Tibshirani R, Friedman J. <em>The Elements of Statistical Learning</em>. 2nd ed. Springer; 2009. Capítulo 7, Model assessment and selection. <a href="https://doi.org/10.1007/978-0-387-84858-7">doi:10.1007/978-0-387-84858-7</a></li>
<li id="r19">Raschka S. Model evaluation, model selection, and algorithm selection in machine learning. <em>arXiv</em>. 2018;arXiv:1811.12808. <a href="https://doi.org/10.48550/arXiv.1811.12808">doi:10.48550/arXiv.1811.12808</a></li>
<li id="r20">Street WN, Wolberg WH, Mangasarian OL. Nuclear feature extraction for breast tumor diagnosis. <em>Proc SPIE</em>. 1993;1905:861–870. <a href="https://doi.org/10.1117/12.148698">doi:10.1117/12.148698</a></li>
<li id="r21">Efron B, Hastie T, Johnstone I, Tibshirani R. Least angle regression. <em>Ann Stat</em>. 2004;32(2):407–499. <a href="https://doi.org/10.1214/009053604000000067">doi:10.1214/009053604000000067</a></li>
<li id="r22">Pedregosa F, Varoquaux G, Gramfort A, et al. Scikit-learn: machine learning in Python. <em>J Mach Learn Res</em>. 2011;12:2825–2830. Sin DOI; <a href="https://jmlr.org/papers/v12/pedregosa11a.html">jmlr.org/papers/v12/pedregosa11a.html</a></li>
<li id="r23">Cawley GC, Talbot NLC. On over-fitting in model selection and subsequent selection bias in performance evaluation. <em>J Mach Learn Res</em>. 2010;11:2079–2107. Sin DOI; <a href="https://jmlr.org/papers/v11/cawley10a.html">jmlr.org/papers/v11/cawley10a.html</a></li>
<li id="r24">Ambroise C, McLachlan GJ. Selection bias in gene extraction on the basis of microarray gene-expression data. <em>Proc Natl Acad Sci USA</em>. 2002;99(10):6562–6566. <a href="https://doi.org/10.1073/pnas.102102699">doi:10.1073/pnas.102102699</a></li>
<li id="r25">Saeb S, Lonini L, Jayaraman A, Mohr DC, Kording KP. The need to approximate the use-case in clinical machine learning. <em>GigaScience</em>. 2017;6(5):gix019. <a href="https://doi.org/10.1093/gigascience/gix019">doi:10.1093/gigascience/gix019</a></li>
<li id="r26">Steyerberg EW, Vickers AJ, Cook NR, et al. Assessing the performance of prediction models: a framework for traditional and novel measures. <em>Epidemiology</em>. 2010;21(1):128–138. <a href="https://doi.org/10.1097/EDE.0b013e3181c30fb2">doi:10.1097/EDE.0b013e3181c30fb2</a></li>
<li id="r27">Riley RD, Ensor J, Snell KIE, et al. Calculating the sample size required for developing a clinical prediction model. <em>BMJ</em>. 2020;368:m441. <a href="https://doi.org/10.1136/bmj.m441">doi:10.1136/bmj.m441</a></li>
<li id="r28">Ramspek CL, Jager KJ, Dekker FW, Zoccali C, van Diepen M. External validation of prognostic models: what, why, how, when and where? <em>Clin Kidney J</em>. 2021;14(1):49–58. <a href="https://doi.org/10.1093/ckj/sfaa188">doi:10.1093/ckj/sfaa188</a></li>
<li id="r29">Brodersen KH, Ong CS, Stephan KE, Buhmann JM. The balanced accuracy and its posterior distribution. In: <em>Proceedings of the 20th International Conference on Pattern Recognition (ICPR 2010)</em>. IEEE; 2010:3121–3124. <a href="https://doi.org/10.1109/ICPR.2010.764">doi:10.1109/ICPR.2010.764</a></li>
<li id="r30">Davis J, Goadrich M. The relationship between precision-recall and ROC curves. In: <em>Proceedings of the 23rd International Conference on Machine Learning (ICML 2006)</em>. ACM; 2006:233–240. <a href="https://doi.org/10.1145/1143844.1143874">doi:10.1145/1143844.1143874</a></li>
<li id="r31">Vickers AJ, Elkin EB. Decision curve analysis: a novel method for evaluating prediction models. <em>Med Decis Making</em>. 2006;26(6):565–574. <a href="https://doi.org/10.1177/0272989X06295361">doi:10.1177/0272989X06295361</a></li>
<li id="r32">Wong A, Otles E, Donnelly JP, et al. External validation of a widely implemented proprietary sepsis prediction model in hospitalized patients. <em>JAMA Intern Med</em>. 2021;181(8):1065–1070. <a href="https://doi.org/10.1001/jamainternmed.2021.2626">doi:10.1001/jamainternmed.2021.2626</a></li>
<li id="r33">Robin X, Turck N, Hainard A, et al. pROC: an open-source package for R and S+ to analyze and compare ROC curves. <em>BMC Bioinformatics</em>. 2011;12:77. <a href="https://doi.org/10.1186/1471-2105-12-77">doi:10.1186/1471-2105-12-77</a></li>
<li id="r34">Kuhn M. Building predictive models in R using the caret package. <em>J Stat Softw</em>. 2008;28(5):1–26. <a href="https://doi.org/10.18637/jss.v028.i05">doi:10.18637/jss.v028.i05</a></li>
<li id="r35">Leisch F, Dimitriadou E. mlbench: Machine Learning Benchmark Problems. Paquete de R. <a href="https://doi.org/10.32614/CRAN.package.mlbench">doi:10.32614/CRAN.package.mlbench</a></li>
<li id="r36">Harrison D, Rubinfeld DL. Hedonic housing prices and the demand for clean air. <em>J Environ Econ Manag</em>. 1978;5(1):81–102. <a href="https://doi.org/10.1016/0095-0696(78)90006-2">doi:10.1016/0095-0696(78)90006-2</a></li>
</ol>

<h4>Libros de texto</h4>
<ul>
<li>Hastie T, Tibshirani R, Friedman J. <em>The Elements of Statistical Learning</em>. 2nd ed. Springer; 2009. — El Capítulo 7 es el tratamiento de referencia de todo lo visto en las Secciones 1–4; PDF gratuito en el sitio de los autores.</li>
<li>James G, Witten D, Hastie T, Tibshirani R. <em>An Introduction to Statistical Learning with Applications in R</em>. 2nd ed. Springer; 2021. — Capítulos 2 y 5 (sesgo–varianza, validación cruzada) a un nivel accesible, con código en R; PDF gratuito en statlearning.com.</li>
<li>Venables WN, Ripley BD. <em>Modern Applied Statistics with S</em> (MASS). 4th ed. Springer; 2002. — La referencia de las funciones de modelado de R usadas aquí (<code>lda</code>, <code>glm</code>, <code>knn</code>).</li>
<li>Steyerberg EW. <em>Clinical Prediction Models</em>. 2nd ed. Springer; 2019. — La perspectiva de la estadística médica sobre validación, calibración e informe de resultados.</li>
<li>Kuhn M, Johnson K. <em>Applied Predictive Modeling</em>. Springer; 2013. — El flujo de trabajo de <code>caret</code> para los pasos de evaluación de este suplemento.</li>
</ul>

<h4>Recursos en línea</h4>
<ul>
<li>Documentación y vignette del paquete pROC — <a href="https://xrobin.github.io/pROC/">xrobin.github.io/pROC</a>; construcción y comparación de curvas ROC/AUC en R.</li>
<li>Documentación del paquete caret, "Model training and tuning" — <a href="https://topepo.github.io/caret/">topepo.github.io/caret</a>; el remuestreo y el ajuste de hiperparámetros empaquetados que se usan ampliamente en R.</li>
<li>Sitio web de la declaración TRIPOD, con listas de verificación y documentos explicativos — <a href="https://www.tripod-statement.org">tripod-statement.org</a>.</li>
<li>Kapoor S, Narayanan A. Leakage model info sheet y lista de verificación — <a href="https://reproducible.cs.princeton.edu">reproducible.cs.princeton.edu</a>.</li>
</ul>
